# Quantum-Inspired Manta Ray Foraging Optimization for Cloud Task Scheduling
### An implementation-first research notebook (QI-MRFO, with QI-DMO as a second host)

**Scope.** Quantum-*inspired* classical optimization for cloud computing. Everything runs on an ordinary CPU with NumPy/SciPy.
No quantum hardware, no quantum circuits, no QPU/annealer access is used or required. The "quantum" content is a
mathematical search mechanism: probability-amplitude registers, Born-rule measurement and a depolarising (decoherence) channel.

**Research loop.** OBSERVE the failure mode of the classical optimizer → HYPOTHESIS → MECHANISM → IMPLEMENT → RUN → ABLATE →
ADVERSARIAL TESTS → INTERPRET. All numbers printed by this notebook are produced when it is executed; the accompanying
report only quotes measured numbers and labels anything else UNMEASURED or HYPOTHETICAL.

**Host algorithm choice (Phase 3).** The mechanism is host-agnostic. The pilot experiments (see the report) showed the
identical mechanism applied to MRFO (Zhao, Zhang & Wang 2020) and to DMO (Agushaka, Ezugwu & Abualigah 2022); MRFO is the
primary host because the effect was largest and the literature gap is documented; DMO is kept as a second host to show transfer.

**Runtime modes.** Set the environment variable `QI_MODE` to `smoke` (≈5 min), `fast` (default, ≈30–60 min on 2 cores) or `full`
(30 seeds, several hours) before starting the kernel. On Linux (Colab, Docker) runs are parallelised with `fork`.

## SECTION 1 — Environment / setup
Installs the few missing packages (Colab already has them), detects the platform, prints versions.

In [ ]:
import sys, os, importlib, subprocess, platform, time
REQUIRED = ["numpy", "scipy", "pandas", "matplotlib"]
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
IN_COLAB = "google.colab" in sys.modules
import numpy, scipy, pandas, matplotlib
print(f"python {platform.python_version()} | numpy {numpy.__version__} | scipy {scipy.__version__} | pandas {pandas.__version__} | matplotlib {matplotlib.__version__}")
print(f"platform: {platform.system()} {platform.machine()} | cores: {os.cpu_count()} | colab: {IN_COLAB}")

## SECTION 2 — Imports and configuration
One configuration dictionary drives every experiment. `QI_MODE` selects the size of the study; nothing else needs editing.

In [ ]:
import numpy as np, time, hashlib
from dataclasses import dataclass, field

import json, math, itertools, warnings, pickle, functools
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")

MODE = os.environ.get("QI_MODE", "fast").lower()
_presets = {
    "smoke": dict(seeds=list(range(2)),  budget=6000,  instances=[(30, 5, "uniform", "high"), (50, 10, "bimodal", "high")], dyn_seeds=list(range(2)), dyn_budget0=6000, dyn_budget=2000),
    "fast":  dict(seeds=list(range(3)),  budget=15000, instances=[(30, 5, "uniform", "high"), (50, 10, "bimodal", "high"), (100, 10, "uniform", "high"), (50, 10, "uniform", "none")], dyn_seeds=list(range(2)), dyn_budget0=12000, dyn_budget=4000),
    "full":  dict(seeds=list(range(30)), budget=20000, instances=[(30, 5, "uniform", "high"), (50, 10, "bimodal", "high"), (100, 10, "uniform", "high"), (50, 10, "uniform", "none"), (200, 20, "uniform", "high"), (100, 10, "lognormal", "low"), (60, 8, "bimodal", "none")], dyn_seeds=list(range(10)), dyn_budget0=15000, dyn_budget=4000),
}
CFG = dict(
    P=30,                       # population size for every population optimizer
    gamma_c_mrfo=1.0,           # decoherence strength gamma = c / n for QI-MRFO   (chosen from the pilot sweep, Section 16 re-checks it)
    gamma_c_dmo=0.25,           # decoherence strength gamma = c / n for QI-DMO
    objective="makespan",       # 'makespan' or 'makespan_energy'
    inst_seed=1,                # seed used to generate the benchmark instances
    parallel=True,              # multiprocessing with fork on Linux; silently sequential elsewhere
    results_dir=os.environ.get("QI_RESULTS_DIR", "results"),
    **_presets[MODE],
)
# raw results are immutable: never overwrite the committed files of a mode; a re-run goes to a new sub-directory
if "QI_RESULTS_DIR" not in os.environ and os.path.exists(os.path.join(CFG["results_dir"], f"baseline_{MODE}.csv")):
    CFG["results_dir"] = os.path.join(CFG["results_dir"], f"rerun_{MODE}_{time.strftime('%Y%m%d_%H%M%S')}")
    print("committed results for this mode exist -> writing this run to", CFG["results_dir"])
os.makedirs(CFG["results_dir"], exist_ok=True)
print(f"mode = {MODE}: seeds={len(CFG['seeds'])}, budget={CFG['budget']} evaluations, instances={len(CFG['instances'])}")

## SECTION 3 — Random-seed control
Every optimizer receives an explicit `seed` and builds its own `numpy.random.default_rng(seed)`. Instances are generated from
`CFG['inst_seed']`, so instance *k* is identical for every algorithm and every run seed. Re-running the notebook with the same
mode reproduces the same numbers (up to floating-point associativity in NumPy reductions).

In [ ]:
GLOBAL_SEED = 2026
np.random.seed(GLOBAL_SEED)          # only affects legacy global calls; all optimizers use per-run Generators
def rng_for(seed): return np.random.default_rng(seed)
assert rng_for(3).random() == rng_for(3).random(), "Generator must be deterministic for a fixed seed"
print("seed control OK")

## SECTION 4 — Cloud scheduling problem representation

**Model (CloudSim-style independent-task scheduling).** $n$ tasks (cloudlets) with lengths $L_i$ (million instructions) are
assigned to $m$ heterogeneous VMs with speeds $S_j$ (MIPS). The execution-time matrix is $ET_{ij} = L_i / S_j$.
A schedule is an integer vector $a \in \{0,\dots,m-1\}^n$ (task $i$ runs on VM $a_i$). The load of VM $j$ is
$\mathrm{Load}_j(a) = \sum_{i: a_i = j} ET_{ij}$ and the **makespan** is $C_{\max}(a) = \max_j \mathrm{Load}_j(a)$.

**Lower bound** used to express results as a *gap*: $LB = \max\!\left(\frac{\sum_i L_i}{\sum_j S_j},\ \frac{\max_i L_i}{\max_j S_j}\right)$ (Q||Cmax bound).

**Encodings.**
* Classical swarm schedulers use a *continuous* position $x \in [0,m)^n$ and the floor decoder $a_i = \lfloor x_i \rfloor$ (`decode`). This is the encoding used by the large majority of PSO/DMO/MRFO cloud-scheduling papers.
* The quantum-inspired version (Section 8) replaces the position by an $n \times m$ **amplitude register** whose *measurement* yields $a$.

In [ ]:
@dataclass
class CloudInstance:
    """Independent tasks (cloudlets) on heterogeneous VMs: the standard CloudSim-style model."""
    task_len: np.ndarray      # MI per task
    vm_mips: np.ndarray       # MIPS per VM
    vm_p_idle: np.ndarray     # W
    vm_p_max: np.ndarray      # W
    name: str = ""
    et: np.ndarray = field(init=False)   # execution-time matrix n x m: et[i, j] = task_len[i] / vm_mips[j]
    def __post_init__(self):
        self.et = self.task_len[:, None] / self.vm_mips[None, :]
    @property
    def n(self): return len(self.task_len)
    @property
    def m(self): return len(self.vm_mips)
    def lower_bound(self):
        # Q||Cmax lower bound: max(total work / total speed, largest task on fastest VM)
        return max(self.task_len.sum() / self.vm_mips.sum(), self.task_len.max() / self.vm_mips.max())
    def lower_bound_pmtn(self):
        """Tighter valid bound: the optimal PREEMPTIVE makespan Q|pmtn|Cmax (Liu & Yang 1974; Horvath, Lam & Sethi 1977;
        Gonzalez & Sahni 1978) = max( max_{k<K} P_k / S_k , P_n / S_K ), K = min(n, m), P_k = sum of the k largest task
        lengths, S_k = sum of the k fastest speeds. Always >= lower_bound(); used for the V5 'gap2' columns."""
        P = np.cumsum(np.sort(self.task_len)[::-1]); S = np.cumsum(np.sort(self.vm_mips)[::-1]); K = min(self.n, self.m)
        prefix = (P[:K - 1] / S[:K - 1]).max() if K > 1 else 0.0
        return max(prefix, P[-1] / S[K - 1])
    def copy_with(self, task_len=None, vm_mips=None, vm_p_idle=None, vm_p_max=None, name=None):
        return CloudInstance(self.task_len if task_len is None else task_len, self.vm_mips if vm_mips is None else vm_mips,
                             self.vm_p_idle if vm_p_idle is None else vm_p_idle, self.vm_p_max if vm_p_max is None else vm_p_max,
                             name or self.name)

def decode(x, m):
    """continuous position in [0, m) -> integer VM index (floor encoding used by most swarm schedulers)"""
    return np.clip(np.floor(x), 0, m - 1).astype(int)

## SECTION 5 — Task / VM generation
Task lengths: `uniform` U(1000, 10000) MI; `bimodal` (80 % small U(500, 3000), 20 % large U(20000, 40000) — a heavy tail as in
cloud traces); `lognormal`; `identical`. VM speeds: `high` heterogeneity (250–2000 MIPS as in the CloudSim examples), `low`, or
`none` (homogeneous). A linear power model $P_j(u) = P^{idle}_j + (P^{max}_j - P^{idle}_j)\,u$ with $P^{idle} = 0.6\,P^{max}$ supports the energy objective.

In [ ]:
def sample_task_lengths(rng, n_tasks, task_dist):
    if task_dist == "uniform":
        return rng.uniform(1000, 10000, n_tasks)
    if task_dist == "lognormal":
        return rng.lognormal(np.log(3000), 1.0, n_tasks)
    if task_dist == "bimodal":          # 80% small, 20% big (heavy tail typical of cloud traces)
        big = rng.random(n_tasks) < 0.2
        return np.where(big, rng.uniform(20000, 40000, n_tasks), rng.uniform(500, 3000, n_tasks))
    if task_dist == "identical":
        return np.full(n_tasks, 5000.0)
    raise ValueError(task_dist)

def sample_vm_speeds(rng, n_vms, hetero):
    if hetero == "high":
        return rng.choice([250, 500, 1000, 1500, 2000], n_vms).astype(float) + rng.uniform(-50, 50, n_vms)
    if hetero == "low":
        return rng.uniform(900, 1100, n_vms)
    if hetero == "none":
        return np.full(n_vms, 1000.0)
    raise ValueError(hetero)

def make_instance(n_tasks, n_vms, seed, task_dist="uniform", hetero="high", name=None):
    rng = np.random.default_rng(seed)
    L = sample_task_lengths(rng, n_tasks, task_dist)
    S = sample_vm_speeds(rng, n_vms, hetero)
    p_max = 100.0 + 0.1 * S             # W, grows with speed
    p_idle = 0.6 * p_max
    return CloudInstance(L, S, p_idle, p_max, name or f"n{n_tasks}_m{n_vms}_{task_dist}_{hetero}_s{seed}")

_demo = make_instance(8, 3, seed=CFG["inst_seed"])
print(_demo.name, "| LB =", round(_demo.lower_bound(), 3))
pd.DataFrame(_demo.et, columns=[f"VM{j} ({_demo.vm_mips[j]:.0f} MIPS)" for j in range(_demo.m)], index=[f"task{i} ({_demo.task_len[i]:.0f} MI)" for i in range(_demo.n)]).round(2)

## SECTION 6 — Objective function
`Objective(inst, kind)` evaluates an integer schedule and **counts evaluations**: the evaluation count is the budget unit shared by
every optimizer (equal-budget fairness). Two objective kinds:

* `makespan`: $f(a) = C_{\max}(a)$ (primary; the landscape is a max-of-sums with plateaus).
* `makespan_energy`: $f(a) = (1-w)\,\frac{C_{\max}(a)}{C^{ref}_{\max}} + w\,\frac{E(a)}{E^{ref}}$ with
  $E(a) = \sum_j \big[P^{idle}_j\,C_{\max}(a) + (P^{max}_j - P^{idle}_j)\,\mathrm{Load}_j(a)\big]/3600$ Wh
  (VMs draw idle power until the last task finishes); the reference values come from a round-robin schedule. This smoother, additive objective is one of the adversarial cases in Section 17.

In [ ]:
class Objective:
    """Evaluate integer assignment vectors (task -> VM). Counts evaluations (the budget unit for every optimizer).
    kind='makespan_migration' (V5, H8): makespan x (1 + lam * voluntary migrations / eligible tasks), where a migration is
    an eligible task (mig_mask: persistent, VM survived) assigned differently from the deployed reference schedule `ref`."""
    def __init__(self, inst: CloudInstance, kind="makespan", w_energy=0.3, ref=None, mig_mask=None, lam=0.0):
        self.inst, self.kind, self.w_energy = inst, kind, w_energy
        if kind == "makespan_migration":
            self.ref = np.asarray(ref); self.mig_mask = np.asarray(mig_mask, bool); self.lam = float(lam)
            self.n_eligible = max(1, int(self.mig_mask.sum()))
        self.n_evals = 0
        self._ar = np.arange(inst.n)
        rr = self._ar % inst.m                      # round-robin reference schedule for scaling
        self.ms_ref, self.e_ref = self._raw(rr)
    def _raw(self, assign):
        et_sel = self.inst.et[self._ar, assign]
        loads = np.bincount(assign, weights=et_sel, minlength=self.inst.m)
        ms = loads.max()
        energy = (self.inst.vm_p_idle * ms + (self.inst.vm_p_max - self.inst.vm_p_idle) * loads).sum() / 3600.0  # Wh
        return ms, energy
    def __call__(self, assign):
        self.n_evals += 1
        ms, en = self._raw(assign)
        if self.kind == "makespan":
            return ms
        if self.kind == "makespan_energy":
            return (1 - self.w_energy) * ms / self.ms_ref + self.w_energy * en / self.e_ref
        if self.kind == "makespan_migration":
            return ms * (1.0 + self.lam * self.migrations(assign) / self.n_eligible)
        raise ValueError(self.kind)
    def migrations(self, assign):
        return int(((np.asarray(assign) != self.ref) & self.mig_mask).sum())
    def details(self, assign):
        ms, en = self._raw(assign)
        d = {"makespan": ms, "energy_Wh": en}
        if self.kind == "makespan_migration": d["migrations"] = self.migrations(assign)
        return d

_obj = Objective(_demo)
_a = np.array([0, 1, 2, 0, 1, 2, 0, 1])
print("round-robin makespan:", round(_obj(_a), 3), "| details:", {k: round(v, 3) for k, v in _obj.details(_a).items()}, "| evaluations so far:", _obj.n_evals)

## SECTION 7 — Classical PSO / DMO / MRFO implementations (+ GA, random, list heuristics)

All population optimizers use the same floor encoding, the same evaluation budget and the same instrumentation
(`Tracker`: best-so-far curve, population diversity as mean pairwise Hamming distance / $n$, the number of tasks changed by
every candidate ("move size"), and the fraction of candidates that are *wasted* (identical schedule to the parent), *neutral*,
*improving* or *worse*).

### 7.1 MRFO — original equations (Zhao, Zhang & Wang, 2020)
With $r, r_1, r_2, r_3 \sim U(0,1)$, $t$ the iteration, $T$ the iteration budget:

* **Chain foraging** $\;x_i^{t+1} = x_i^t + r\,(x_{i-1}^t - x_i^t) + \alpha\,(x_{best}^t - x_i^t),\quad \alpha = 2r\sqrt{|\log r|}$ (for $i=1$ the predecessor is $x_{best}$).
* **Cyclone foraging** $\;x_i^{t+1} = x_{best} + r\,(x_{i-1}^t - x_i^t) + \beta\,(x_{best} - x_i^t),\quad \beta = 2e^{r_1 (T-t+1)/T}\sin(2\pi r_1)$;
  while $t/T < \mathrm{rand}$ a random position $x_{rand}$ replaces $x_{best}$ (exploration).
* **Somersault foraging** $\;x_i^{t+1} = x_i^t + S\,(r_2\,x_{best} - r_3\,x_i^t),\quad S = 2$.
* Greedy replacement after each phase; per iteration each individual is evaluated twice.

### 7.2 DMO — original equations (Agushaka, Ezugwu & Abualigah, 2022; MATLAB/MEALPY form)
* Alpha selection by roulette on $\phi_i = e^{-f_i/\bar f}$.
* **Alpha group**: candidate $X_{cand} = X_\alpha + \varphi \odot (X_\alpha - X_k)$, $\varphi \sim \tfrac{peep}{2}\,U(-1,1)^n$, accepted if better, else the counter $C_i$ grows.
* **Scouts**: $X_{cand} = X_i + \varphi \odot (X_i - X_k)$; sleeping mound $sm_i = (f_{cand} - f_i)/\max(f_{cand}, f_i)$.
* **Babysitter exchange**: the first $B$ mongooses are re-initialised uniformly when $C_i \ge L = 0.6\,n\,B$.
* **Next position**: $X_i \leftarrow X_i \mp CF\,\varphi\,r\,(X_i - sm_i)$ with $CF = (1 - t/T)^{2t/T}$, sign set by the trend of the average sleeping mound; this move is **unconditional** in the authors' code (kept faithfully; `greedy_next=True` is the MEALPY "developed" variant).

### 7.3 PSO, GA and heuristics
Inertia-weight PSO ($w=0.729$, $c_1=c_2=1.49445$); a discrete GA (tournament-2, uniform crossover, $1/n$ reassignment mutation, elitism) as an independent classical baseline that works natively in the assignment space; random search; Min-Min and Max-Min list-scheduling heuristics; a 1-move hill climber for landscape probes.

In [ ]:
class Tracker:
    """Records per-iteration statistics common to all optimizers (used to test the MECHANISM, not just the score).
    With `inst` given (V5), two extra diagnostics are recorded per candidate: whether the schedule was ever evaluated
    before in the run (global duplicate; 'wasted' only counts parent-identical candidates) and whether the candidate
    moves a task off its reference schedule's critical VM (a necessary condition for a strict makespan improvement).
    They only read the candidates, so the optimizers' behaviour is unchanged."""
    def __init__(self, n, m, inst=None):
        self.n, self.m = n, m
        self.evals, self.best, self.mean, self.div_ham = [], [], [], []
        self.wasted = self.neutral = self.improving = self.worse = 0
        self.move_sizes = []
        self.gb_improvements = 0
        self.dup_global = self.touch_crit = self.x_cands = self.x_improving = 0
        self._et = None if inst is None else inst.et
        self._seen = set(); self._ar = np.arange(n)
        self.flags = []            # per candidate: 1 = improving, 0 = not (used for late-run improvement rates)
        self.t0 = time.time()
    def candidate(self, parent_assign, child_assign, f_parent, f_child):
        diff = parent_assign != child_assign
        changed = int(diff.sum())
        self.move_sizes.append(changed)
        if changed == 0: self.wasted += 1
        elif f_child < f_parent - 1e-12: self.improving += 1
        elif abs(f_child - f_parent) <= 1e-12: self.neutral += 1
        else: self.worse += 1
        if self._et is not None:
            self.flags.append(int(changed > 0 and f_child < f_parent - 1e-12))
            self._seen.add(hashlib.blake2b(parent_assign.tobytes(), digest_size=8).digest())
            key = hashlib.blake2b(child_assign.tobytes(), digest_size=8).digest()   # deterministic (no PYTHONHASHSEED salt)
            if key in self._seen: self.dup_global += 1
            else: self._seen.add(key)
            loads = np.bincount(parent_assign, weights=self._et[self._ar, parent_assign], minlength=self.m)
            crit = loads >= loads.max() * (1 - 1e-12)
            self.touch_crit += int((diff & crit[parent_assign]).any())
    def snapshot(self, n_evals, assigns, fits, best):
        self.evals.append(n_evals); self.best.append(best); self.mean.append(float(np.mean(fits)))
        A = np.asarray(assigns); P = len(A)
        if P > 1:
            ham = 0.0
            for i in range(P - 1):
                ham += (A[i + 1:] != A[i]).sum()
            self.div_ham.append(ham / (P * (P - 1) / 2) / self.n)
        else:
            self.div_ham.append(0.0)
    def summary(self):
        tot = max(1, self.wasted + self.neutral + self.improving + self.worse)
        return {"wasted_frac": self.wasted / tot, "neutral_frac": self.neutral / tot,
                "improving_frac": self.improving / tot, "worse_frac": self.worse / tot,
                "mean_move_size": float(np.mean(self.move_sizes)) if self.move_sizes else 0.0,
                "gb_impr_per_1k": 1000.0 * self.gb_improvements / max(1, self.evals[-1] if self.evals else 1),
                "runtime_s": time.time() - self.t0}
    def summary_v5(self):
        """V5 diagnostics (needs Tracker(..., inst)): global duplicates, critical-VM touches, exchange-move counts,
        late-half improvement rate and the budget fraction at which the global best last improved."""
        tot = max(1, len(self.move_sizes))
        ev, bs = np.asarray(self.evals, float), np.asarray(self.best, float)
        drops = np.where(np.diff(bs) < 0)[0]
        last = ev[drops[-1] + 1] / ev[-1] if len(drops) and ev[-1] > 0 else 0.0
        fl = np.asarray(self.flags)
        return {"dup_global_frac": self.dup_global / tot, "touch_crit_frac": self.touch_crit / tot,
                "x_frac": self.x_cands / tot, "x_success": self.x_improving / max(1, self.x_cands),
                "late_improving_frac": float(fl[len(fl) // 2:].mean()) if len(fl) > 1 else 0.0,
                "last_gb_impr_frac": float(last)}

In [ ]:
def _list_schedule(inst, pick):
    n, m = inst.n, inst.m
    ready = np.zeros(m); assign = -np.ones(n, int); left = set(range(n))
    while left:
        idx = np.array(sorted(left))
        ct = inst.et[idx] + ready[None, :]
        best_vm = ct.argmin(1); best_ct = ct[np.arange(len(idx)), best_vm]
        k = pick(best_ct); t = idx[k]; v = best_vm[k]
        assign[t] = v; ready[v] = best_ct[k]; left.remove(t)
    return assign

def min_min(inst): return _list_schedule(inst, np.argmin)
def max_min(inst): return _list_schedule(inst, np.argmax)

def local_search_1move(inst, assign, max_steps=10000):
    """Best-improvement hill climbing: move one task off the bottleneck VM."""
    a = assign.copy(); ar = np.arange(inst.n)
    for _ in range(max_steps):
        loads = np.bincount(a, weights=inst.et[ar, a], minlength=inst.m)
        ms = loads.max(); b = loads.argmax()
        best = (ms, None)
        for t in np.where(a == b)[0]:
            for v in range(inst.m):
                if v == b: continue
                new_b = ms - inst.et[t, b]; new_v = loads[v] + inst.et[t, v]
                others = np.delete(loads, [b, v]).max() if inst.m > 2 else -np.inf
                new_ms = max(new_b, new_v, others)
                if new_ms < best[0] - 1e-9: best = (new_ms, (t, v))
        if best[1] is None: return a
        t, v = best[1]; a[t] = v
    return a

def critical_exchange(inst, a, rng):
    """V5 critical exchange move (CXM). t is drawn uniformly from the tasks on the critical VM(s) of schedule a;
    u uniformly from the tasks on other VMs that are SHORTER than t (necessary for the exchange to lower the critical
    load); t and u swap VMs. If no shorter task exists elsewhere, t is relocated to a uniformly random other VM.
    Returns (new_a, t, u), u = -1 for a relocation. Shared by QI-MRFO and the GA so that both get the identical operator."""
    n, m = inst.n, inst.m
    new = a.copy()
    if m < 2: return new, -1, -1
    loads = np.bincount(a, weights=inst.et[np.arange(n), a], minlength=m)
    crit = loads >= loads.max() * (1 - 1e-12)
    T = np.flatnonzero(crit[a])
    t = T[rng.integers(len(T))]; b = a[t]
    U = np.flatnonzero((a != b) & (inst.task_len < inst.task_len[t]))
    if len(U):
        u = U[rng.integers(len(U))]
        new[t], new[u] = a[u], b
        return new, t, u
    v = rng.integers(m - 1); v += int(v >= b)
    new[t] = v
    return new, t, -1

def count_improving_moves(inst, a, eps=1e-9):
    """Landscape diagnostic at schedule a: number of STRICTLY makespan-improving (i) relocations of a critical task and
    (ii) swaps of a critical task with a task on another VM. With several VMs tied at the makespan no single move can
    improve it: returns (0, 0, n_tied). Vectorised version of observe_v5_localopt.improving_moves."""
    n, m, et = inst.n, inst.m, inst.et
    L = np.bincount(a, weights=et[np.arange(n), a], minlength=m); ms = L.max()
    tied = int((L >= ms * (1 - 1e-12)).sum())
    if tied > 1 or m < 2: return 0, 0, tied
    b = int(L.argmax()); Tb = np.flatnonzero(a == b); Ub = np.flatnonzero(a != b)
    rest = np.full(m, -np.inf)                                   # rest[v] = max load over VMs other than b and v
    for v in range(m):
        if v != b:
            mask = np.ones(m, bool); mask[[b, v]] = False
            rest[v] = L[mask].max() if mask.any() else -np.inf
    V = np.delete(np.arange(m), b)
    rel_new = np.maximum(np.maximum((ms - et[Tb, b])[:, None], L[V][None, :] + et[np.ix_(Tb, V)]), rest[V][None, :])
    n_rel = int((rel_new < ms - eps).sum())
    vu = a[Ub]
    nb = ms - et[Tb, b][:, None] + et[Ub, b][None, :]
    nv = (L[vu] - et[Ub, vu])[None, :] + et[np.ix_(Tb, np.arange(m))][:, vu]
    n_swap = int((np.maximum(np.maximum(nb, nv), rest[vu][None, :]) < ms - eps).sum())
    return n_rel, n_swap, 1

In [ ]:
# All optimizers share the signature (inst, obj, budget, P, seed, ...) and return a dict with best_f, best_assign,
# tracker and a 'state' that can be passed back as init_state for warm-started (dynamic) re-optimization.
def _init_pop(rng, P, n, m):
    return rng.uniform(0, m, (P, n))

def run_pso(inst, obj, budget, P=30, seed=0, w=0.729, c1=1.49445, c2=1.49445, track=True, init_state=None):
    """Standard inertia-weight PSO (Clerc constriction values), floor decoder."""
    rng = np.random.default_rng(seed); n, m = inst.n, inst.m
    tr = Tracker(n, m, inst)
    if init_state is None:
        X = _init_pop(rng, P, n, m); V = rng.uniform(-1, 1, (P, n))
    else:
        X = np.clip(init_state["X"].copy(), 0, m - 1e-9); V = init_state["V"].copy()
    A = decode(X, m); F = np.array([obj(a) for a in A])
    pb, pbF, pbA = X.copy(), F.copy(), A.copy()
    g = pbF.argmin(); gb, gbF, gbA = pb[g].copy(), pbF[g], pbA[g].copy()
    vmax = m / 2
    tr.snapshot(obj.n_evals, A, F, gbF)
    while obj.n_evals + P <= budget:
        r1, r2 = rng.random((P, n)), rng.random((P, n))
        V = np.clip(w * V + c1 * r1 * (pb - X) + c2 * r2 * (gb - X), -vmax, vmax)
        X = np.clip(X + V, 0, m - 1e-9)
        A_new = decode(X, m)
        for i in range(P):
            f = obj(A_new[i])
            if track: tr.candidate(pbA[i], A_new[i], pbF[i], f)
            if f < pbF[i]:
                pb[i], pbF[i], pbA[i] = X[i].copy(), f, A_new[i].copy()
                if f < gbF: gb, gbF, gbA = X[i].copy(), f, A_new[i].copy(); tr.gb_improvements += 1
        tr.snapshot(obj.n_evals, pbA, pbF, gbF)
    return {"best_f": gbF, "best_assign": gbA, "tracker": tr, "state": {"X": X, "V": V}}

def run_dmo(inst, obj, budget, P=30, seed=0, n_baby_sitter=3, peep=2.0, track=True, greedy_next=False, init_state=None):
    """Dwarf Mongoose Optimization, faithful to the authors' MATLAB code as reproduced in MEALPY OriginalDMOA.
    greedy_next=True replaces the unconditional 'next mongoose position' move by a greedy one (MEALPY DevDMOA style)."""
    rng = np.random.default_rng(seed); n, m = inst.n, inst.m
    tr = Tracker(n, m, inst)
    X = _init_pop(rng, P, n, m) if init_state is None else np.clip(init_state["X"].copy(), 0, m - 1e-9)
    A = decode(X, m); F = np.array([obj(a) for a in A])
    C = np.zeros(P); tau = -np.inf; L = np.round(0.6 * n * n_baby_sitter)
    g = F.argmin(); gbF, gbA = F[g], A[g].copy()
    max_iter = max(1, (budget - P) // (3 * P))
    tr.snapshot(obj.n_evals, A, F, gbF)
    def clip(x): return np.clip(x, 0, m - 1e-9)
    for it in range(1, max_iter + 1):
        if obj.n_evals + 3 * P + n_baby_sitter > budget: break         # never exceed the evaluation budget
        CF = (1.0 - it / max_iter) ** (2.0 * it / max_iter)
        fi = np.exp(-F / F.mean()); prob = fi / fi.sum()
        for i in range(P):                                             # alpha-group foraging
            alpha = rng.choice(P, p=prob)
            k = rng.choice([k for k in range(P) if k not in (i, alpha)])
            phi = (peep / 2) * rng.uniform(-1, 1, n)
            new = clip(X[alpha] + phi * (X[alpha] - X[k]))
            a_new = decode(new, m); f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f)
            if f < F[i]: X[i], A[i], F[i] = new, a_new, f
            else: C[i] += 1
            if f < gbF: gbF, gbA = f, a_new.copy(); tr.gb_improvements += 1
        SM = np.zeros(P)
        for i in range(P):                                             # scout group + sleeping mound
            k = rng.choice([k for k in range(P) if k != i])
            phi = (peep / 2) * rng.uniform(-1, 1, n)
            new = clip(X[i] + phi * (X[i] - X[k]))
            a_new = decode(new, m); f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f)
            SM[i] = (f - F[i]) / max(f, F[i])
            if f < F[i]: X[i], A[i], F[i] = new, a_new, f
            else: C[i] += 1
            if f < gbF: gbF, gbA = f, a_new.copy(); tr.gb_improvements += 1
        for i in range(n_baby_sitter):                                 # babysitter exchange
            if C[i] >= L:
                X[i] = rng.uniform(0, m, n); A[i] = decode(X[i], m); F[i] = obj(A[i]); C[i] = 0
                if F[i] < gbF: gbF, gbA = F[i], A[i].copy(); tr.gb_improvements += 1
        new_tau = SM.mean()
        for i in range(P):                                             # next mongoose position
            phi = (peep / 2) * rng.uniform(-1, 1, n)
            if new_tau > tau: new = X[i] - CF * phi * rng.random() * (X[i] - SM[i])
            else:             new = X[i] + CF * phi * rng.random() * (X[i] - SM[i])
            tau = new_tau
            new = clip(new); a_new = decode(new, m); f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f)
            if (not greedy_next) or f < F[i]: X[i], A[i], F[i] = new, a_new, f
            if f < gbF: gbF, gbA = f, a_new.copy(); tr.gb_improvements += 1
        tr.snapshot(obj.n_evals, A, F, gbF)
    return {"best_f": gbF, "best_assign": gbA, "tracker": tr, "state": {"X": X}}

def run_mrfo(inst, obj, budget, P=30, seed=0, S=2.0, track=True, init_state=None):
    """Manta Ray Foraging Optimization, faithful to Zhao et al. 2020 as implemented in MEALPY OriginalMRFO."""
    rng = np.random.default_rng(seed); n, m = inst.n, inst.m
    tr = Tracker(n, m, inst)
    X = _init_pop(rng, P, n, m) if init_state is None else np.clip(init_state["X"].copy(), 0, m - 1e-9)
    A = decode(X, m); F = np.array([obj(a) for a in A])
    g = F.argmin(); gb, gbF, gbA = X[g].copy(), F[g], A[g].copy()
    T = max(1, (budget - P) // (2 * P))
    tr.snapshot(obj.n_evals, A, F, gbF)
    def clip(x): return np.clip(x, 0, m - 1e-9)
    for t in range(1, T + 1):
        for i in range(P):
            if rng.random() < 0.5:                       # cyclone foraging
                r1 = rng.random()
                beta = 2 * np.exp(r1 * (T - t) / T) * np.sin(2 * np.pi * r1)
                if (t + 1) / T < rng.random():           # exploration around a random reference
                    xr = rng.uniform(0, m, n)
                    if i == 0: new = xr + rng.random() * (xr - X[i]) + beta * (xr - X[i])
                    else:      new = xr + rng.random() * (X[i - 1] - X[i]) + beta * (xr - X[i])
                else:
                    if i == 0: new = gb + rng.random() * (gb - X[i]) + beta * (gb - X[i])
                    else:      new = gb + rng.random() * (X[i - 1] - X[i]) + beta * (gb - X[i])
            else:                                        # chain foraging
                r = rng.random(); alpha = 2 * r * np.sqrt(np.abs(np.log(r)))
                if i == 0: new = X[i] + r * (gb - X[i]) + alpha * (gb - X[i])
                else:      new = X[i] + r * (X[i - 1] - X[i]) + alpha * (gb - X[i])
            new = clip(new); a_new = decode(new, m); f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f)
            if f < F[i]: X[i], A[i], F[i] = new, a_new, f
            if f < gbF: gb, gbF, gbA = new.copy(), f, a_new.copy(); tr.gb_improvements += 1
        for i in range(P):                               # somersault foraging
            new = clip(X[i] + S * (rng.random() * gb - rng.random() * X[i]))
            a_new = decode(new, m); f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f)
            if f < F[i]: X[i], A[i], F[i] = new, a_new, f
            if f < gbF: gb, gbF, gbA = new.copy(), f, a_new.copy(); tr.gb_improvements += 1
        tr.snapshot(obj.n_evals, A, F, gbF)
    return {"best_f": gbF, "best_assign": gbA, "tracker": tr, "state": {"X": X}}

def run_ga(inst, obj, budget, P=30, seed=0, pm=None, track=True, init_state=None, hypermutation=0.0, exchange=0.0, seed_assign=None):
    """Discrete GA baseline: tournament(2), uniform crossover, per-gene reassignment mutation, 1-elitism.
    hypermutation>0: mutation rate x10 during the first `hypermutation` fraction of the budget (Cobb-style).
    exchange>0 (V5 control for H5): each child additionally undergoes the critical exchange move with this probability."""
    rng = np.random.default_rng(seed); n, m = inst.n, inst.m
    pm = pm or 1.0 / n
    tr = Tracker(n, m, inst)
    A = rng.integers(0, m, (P, n)) if init_state is None else np.clip(init_state["A"].copy(), 0, m - 1)
    if seed_assign is not None: A[0] = seed_assign             # V5 (H7): heuristic seeding of one individual
    F = np.array([obj(a) for a in A])
    g = F.argmin(); gbF, gbA = F[g], A[g].copy()
    start = obj.n_evals
    tr.snapshot(obj.n_evals, A, F, gbF)
    while obj.n_evals + P <= budget:
        rate = pm * 10 if (obj.n_evals - start) < hypermutation * (budget - start) else pm
        newA = np.empty_like(A); newF = np.empty_like(F)
        e = F.argmin(); newA[0], newF[0] = A[e], F[e]
        for c in range(1, P):
            i1, j1 = rng.integers(0, P, 2); p1 = i1 if F[i1] <= F[j1] else j1
            i2, j2 = rng.integers(0, P, 2); p2 = i2 if F[i2] <= F[j2] else j2
            child = np.where(rng.random(n) < 0.5, A[p1], A[p2])
            mut = rng.random(n) < rate
            child[mut] = rng.integers(0, m, mut.sum())
            xm = exchange > 0 and rng.random() < exchange
            if xm: child = critical_exchange(inst, child, rng)[0]
            f = obj(child)
            if track:
                tr.candidate(A[p1], child, F[p1], f)
                if xm: tr.x_cands += 1; tr.x_improving += int(f < F[p1] - 1e-12)
            newA[c], newF[c] = child, f
            if f < gbF: gbF, gbA = f, child.copy(); tr.gb_improvements += 1
        A, F = newA, newF
        tr.snapshot(obj.n_evals, A, F, gbF)
    return {"best_f": gbF, "best_assign": gbA, "tracker": tr, "state": {"A": A}}

def run_one_plus_one(inst, obj, budget, P=1, seed=0, decoherence=0.0, exchange=0.0, track=True, init_state=None, seed_assign=None):
    """V5 minimal classical control for H7: a (1+1)-EA whose mutation is exactly what a fully collapsed QI-MRFO+CXM
    register set produces. Each task is re-drawn uniformly over the VMs with probability `decoherence` (the depolarising
    floor acting on a basis state), then the critical exchange is applied with probability `exchange`. Strict
    acceptance, one evaluation per candidate. `seed_assign` / init_state['A'] give the start schedule (default: uniform
    random). P is ignored (kept for the common signature)."""
    rng = np.random.default_rng(seed); n, m = inst.n, inst.m
    tr = Tracker(n, m, inst)
    if seed_assign is not None: a = np.asarray(seed_assign).copy()
    elif init_state is not None and "A" in init_state: a = np.asarray(init_state["A"])[0].copy()
    else: a = rng.integers(0, m, n)
    f = obj(a)
    tr.snapshot(obj.n_evals, [a], [f], f)
    while obj.n_evals + 1 <= budget:
        child = a.copy()
        if decoherence > 0:
            mut = rng.random(n) < decoherence
            child[mut] = rng.integers(0, m, mut.sum())
        xm = exchange > 0 and rng.random() < exchange
        if xm: child = critical_exchange(inst, child, rng)[0]
        fc = obj(child)
        if track:
            tr.candidate(a, child, f, fc)
            if xm: tr.x_cands += 1; tr.x_improving += int(fc < f - 1e-12)
        if fc < f: a, f = child, fc; tr.gb_improvements += 1
        if obj.n_evals % 60 == 0: tr.snapshot(obj.n_evals, [a], [f], f)
    tr.snapshot(obj.n_evals, [a], [f], f)
    return {"best_f": f, "best_assign": a, "tracker": tr, "state": {"A": a[None, :]}}

def run_random(inst, obj, budget, seed=0, P=30, track=True, init_state=None):
    rng = np.random.default_rng(seed); n, m = inst.n, inst.m
    tr = Tracker(n, m, inst); gbF, gbA = np.inf, None
    while obj.n_evals + P <= budget:
        A = rng.integers(0, m, (P, n)); F = np.array([obj(a) for a in A])
        g = F.argmin()
        if F[g] < gbF: gbF, gbA = F[g], A[g].copy(); tr.gb_improvements += 1
        tr.snapshot(obj.n_evals, A, F, gbF)
    return {"best_f": gbF, "best_assign": gbA, "tracker": tr, "state": {}}

ALGOS = {"PSO": run_pso, "DMO": run_dmo, "MRFO": run_mrfo, "GA": run_ga, "Random": run_random, "(1+1)-EA": run_one_plus_one}

## SECTION 8 — The quantum-inspired mechanism

### 8.1 Observation that motivates it (measured in the pilot, reproduced in Section 12)
With the floor encoding, a DMO or MRFO move changes about half of all task assignments per candidate (the moves are
proportional to inter-individual distances in $[0,m)^n$, and rounding turns them into random re-draws), while an improving move
on the assignment landscape changes 1–5 tasks. The result is ≤ 2 % improving candidates and performance at or below random
search for $n \ge 50$. The encoding gives the optimizer no controllable notion of **move size**.

### 8.2 Representation: one probability-amplitude register per task
Individual $i$ holds $\Psi_i \in \mathbb{R}^{n \times m}$, one row per task, each row a **unit vector** (an $m$-level generalisation of a real
Q-bit / "rebit"): $\sum_j \Psi_i[t,j]^2 = 1$. The uniform superposition $\Psi = m^{-1/2}\mathbf{1}$ is the initial state.

**Measurement (Born rule).** A schedule is *measured* by sampling every task independently:
$\Pr(a_t = j) = \Psi[t,j]^2$. One measurement = one objective evaluation. A measured schedule $b$ is encoded back as the
**basis state** $E(b)$ (one-hot rows).

**Purity and move size.** The purity of register $t$ is $\pi_t = \sum_j p_{tj}^2 \in [1/m, 1]$. Two independent measurements of the same
register set differ, in expectation, in $\;n - \sum_t \pi_t\;$ tasks. Purity therefore *is* the move-size control that the floor encoding lacks:
concentrated registers make small moves, diffuse registers make large ones, and the host algorithm's own dynamics move purity continuously.

### 8.3 Host dynamics applied unchanged to amplitudes
Every MRFO/DMO equation of Section 7 is applied to the amplitude matrices, followed by row renormalisation
$\Pi(\Psi)[t,:] = \Psi[t,:]/\|\Psi[t,:]\|$ (the projection back onto the state space). **Attractors are basis states**: the best-known schedule enters
the equations as $E(b_{best})$ (MRFO) or the alpha's measured schedule $E(b_\alpha)$ (DMO: "the alpha peeps its food position").
Section 15 shows that using the alpha's superposition instead of its measured schedule destroys the search — collapse-conditioned attraction is essential.

### 8.4 Decoherence: the depolarising channel
After every update the candidate register passes through the depolarising channel
$$\mathcal{D}_\gamma(p) = (1-\gamma)\,p + \gamma\,\tfrac{1}{m}\mathbf{1}, \qquad \Psi \leftarrow \operatorname{sign}(\Psi)\sqrt{\mathcal{D}_\gamma(\Psi^2)} .$$
It bounds purity away from 1: with $\gamma = c/n$ a fully collapsed register keeps an expected $\approx 2c\,(1-1/m)$ random reassignments per measurement,
i.e. a controllable floor on move size ("the environment never lets the swarm fully collapse"). With $\gamma = 1$ it is the classical random reset
(DMO's babysitter exchange); with intermediate $\gamma$ it is **controlled forgetting**, used in Section 17 for dynamic workloads.

### 8.5 What is quantum-inspired, what is its classical equivalent
| Ingredient | Quantum origin | Classical equivalent (the *twin*, `mode='linear'`) |
|---|---|---|
| Amplitude register, Born rule $p = \psi^2$ | measurement postulate | probability vector with linear normalisation |
| Signed amplitudes (can cancel when mixed) | interference | none (probabilities are non-negative) |
| Basis-state attractor | measurement collapse | one-hot probability vector |
| Depolarising channel | decoherence / noise channel | mutation floor / partial re-initialisation (PBIL-style) |
| Column deletion at VM failure | projective measurement | delete + renormalise |

The classical twin keeps everything except the squaring and the signs. Whether the *quantum* part (squaring, sign) matters beyond
the *representational* part (superposition + measurement + decoherence) is exactly what the ablation in Section 15 tests.
This is **not** quantum computing: no qubits are simulated as a joint state, no unitary evolution, no entanglement; the registers are $n$ independent
$m$-dimensional real unit vectors updated by classical arithmetic in $O(nm)$ per candidate.

### 8.6 Complexity
Per candidate: $O(nm)$ arithmetic + one $O(nm)$ measurement (cumulative sums) + one $O(n)$ objective evaluation, versus $O(n)$ for the floor encoding.
Memory: $P\,n\,m$ floats (e.g. $30 \times 100 \times 10 \times 8$ B = 240 kB). Section 17 measures the actual overhead.

In [ ]:
def born_probs(psi):
    p = psi ** 2
    return p / p.sum(1, keepdims=True)

def renormalise(psi, m):
    nrm = np.sqrt((psi ** 2).sum(1, keepdims=True))
    bad = (nrm[:, 0] < 1e-12)
    psi = psi / np.where(nrm < 1e-12, 1.0, nrm)
    if bad.any():
        psi[bad] = 1.0 / np.sqrt(m)
    return psi

def probs(psi, mode):
    if mode.startswith("born"):
        return born_probs(psi)
    p = np.clip(psi, 0, None); return p / np.maximum(p.sum(1, keepdims=True), 1e-300)

def measure(psi, rng, mode):
    """Sample one schedule from the register set (Born rule or linear probabilities)."""
    p = probs(psi, mode)
    u = rng.random(len(p))
    a = (np.cumsum(p, 1) < u[:, None]).sum(1)
    return np.minimum(a, p.shape[1] - 1)

def basis_state(assign, m):
    E = np.zeros((len(assign), m)); E[np.arange(len(assign)), assign] = 1.0
    return E          # a one-hot row is both a unit amplitude vector and a probability vector

def project(psi, m, mode):
    """Bring an updated register set back to a valid state."""
    if mode == "born_signed":
        return renormalise(psi, m)
    if mode == "born_abs":
        return renormalise(np.abs(psi), m)
    p = np.clip(psi, 0, None); s = p.sum(1, keepdims=True)
    return np.where(s > 1e-12, p / np.where(s > 1e-12, s, 1.0), 1.0 / m)

def purity(psi, mode):
    return (probs(psi, mode) ** 2).sum(1)            # per task; 1 = collapsed, 1/m = uniform

def depolarise(psi, gamma, m, mode):
    """Depolarising channel: p <- (1-gamma) p + gamma/m ; signs kept for signed amplitudes."""
    if gamma <= 0: return psi
    if mode.startswith("born"):
        p2 = (1 - gamma) * born_probs(psi) + gamma / m
        return np.sign(psi + (psi == 0)) * np.sqrt(p2)
    return (1 - gamma) * psi + gamma / m

def uniform_state(n, m, mode):
    return np.full((n, m), 1.0 / np.sqrt(m)) if mode.startswith("born") else np.full((n, m), 1.0 / m)

# --- state transformations used by the dynamic-workload experiments ---------------------------------
def shock_state(state, gamma, mode):
    """Decoherence shock: apply the depolarising channel of strength gamma to every register of every individual."""
    Psi = state["Psi"]; m = Psi.shape[2]
    return {"Psi": np.stack([depolarise(Psi[i], gamma, m, mode) for i in range(len(Psi))])}

def remove_vm_state(state, j, mode):
    """VM j disappears: delete its column (projective measurement onto the surviving VMs) and renormalise."""
    Psi = np.delete(state["Psi"], j, axis=2); m = Psi.shape[2]
    return {"Psi": np.stack([project(Psi[i], m, mode) for i in range(len(Psi))])}

def add_vm_state(state, mode):
    """A new VM appears: give it the amplitude of a uniform share (1/(m+1)) in every register."""
    Psi = state["Psi"]; P, n, m = Psi.shape
    p = np.stack([probs(Psi[i], mode) for i in range(P)]) * (m / (m + 1.0))
    p = np.concatenate([p, np.full((P, n, 1), 1.0 / (m + 1.0))], axis=2)
    if mode.startswith("born"):
        return {"Psi": np.sqrt(p)}
    return {"Psi": p}

def add_tasks_state(state, idx_new, mode):
    """Tasks idx_new were replaced by new arrivals: their registers are reset to the uniform superposition."""
    Psi = state["Psi"].copy(); m = Psi.shape[2]
    Psi[:, idx_new, :] = uniform_state(len(idx_new), m, mode)[None]
    return {"Psi": Psi}

def collapse_rows(psi, rows, outcomes, gamma, m, mode):
    """Measurement back-action (V5): the registers `rows` collapse onto the measured VMs `outcomes`, then pass through the
    same depolarising floor as every candidate register (so a collapsed row keeps purity < 1 when gamma > 0)."""
    psi = psi.copy()
    psi[rows] = depolarise(basis_state(np.asarray(outcomes), m), gamma, m, mode)
    return psi

# --- demonstration of the purity / move-size relation and of Born-vs-linear mixing -------------------------------
_rng = np.random.default_rng(0); _m = 10
_cs = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
print("expected tasks changed per measurement of a COLLAPSED register set (n=100) after the channel gamma=c/n:")
for c in _cs:
    E = basis_state(_rng.integers(0, _m, 100), _m)
    D = depolarise(E, c / 100, _m, "born_signed")
    print(f"  c={c:4.2f}: purity={purity(D, 'born_signed').mean():.4f}  E[changed tasks]={100 - purity(D, 'born_signed').sum():.2f}")
phi = np.linspace(-1, 1, 9)
print("\nprobability of the attractor's VM after mixing E + phi*(E - U) with a UNIFORM register U (m=10):")
for f in phi:
    e = np.zeros((1, _m)); e[0, 0] = 1
    born = born_probs(renormalise(e + f * (e - np.full((1, _m), 1 / np.sqrt(_m))), _m))[0, 0]
    lin = project(e + f * (e - np.full((1, _m), 1 / _m)), _m, "linear")[0, 0]
    print(f"  phi={f:+.2f}: Born={born:.3f}  linear={lin:.3f}")

## SECTION 9 — Quantum-inspired algorithm implementation (QI-MRFO and QI-DMO)

**Modified equations (MRFO → QI-MRFO).** Replace $x_i$ by $\Psi_i$, $x_{best}$ by $E(b_{best})$, $x_{rand}$ by the basis state of a random schedule, and wrap every update as
$\Psi_{cand} = \mathcal{D}_\gamma\big(\Pi(\cdot)\big)$, then *measure* $\Psi_{cand}$ to obtain the candidate schedule that is evaluated and compared greedily:

* chain: $\Psi_{cand} = \mathcal{D}_\gamma\Pi\big[\Psi_i + r(\Psi_{i-1} - \Psi_i) + \alpha(E(b_{best}) - \Psi_i)\big]$
* cyclone: $\Psi_{cand} = \mathcal{D}_\gamma\Pi\big[E(b_{best}) + r(\Psi_{i-1} - \Psi_i) + \beta(E(b_{best}) - \Psi_i)\big]$ (or $E(b_{rand})$ during exploration)
* somersault: $\Psi_{cand} = \mathcal{D}_\gamma\Pi\big[\Psi_i + S(r_2 E(b_{best}) - r_3 \Psi_i)\big]$ — a reflection-like move about the best basis state.

**Modified equations (DMO → QI-DMO).** Alpha group: $\Psi_{cand} = \mathcal{D}_\gamma\Pi[E(b_\alpha) + \varphi \odot (E(b_\alpha) - \Psi_k)]$; scouts:
$\mathcal{D}_\gamma\Pi[\Psi_i + \varphi \odot (\Psi_i - \Psi_k)]$; babysitter exchange: $\Psi_i \leftarrow \mathcal{D}_{\gamma_{reset}}(\Psi_i)$ ($\gamma_{reset}=1$ reproduces the classical reset);
next position: $\mathcal{D}_\gamma\Pi[\Psi_i \mp CF\,\varphi\,r\,(\Psi_i - sm_i)]$ (unconditional, as in the original). $\varphi$ is drawn per task (per row).

**State-transition process.** $(\Psi_i, b_i, f_i) \to$ candidate register $\to$ measurement $\to$ evaluation $\to$ greedy acceptance of $(\Psi_{cand}, a_{cand}, f_{cand})$.
The identity of the host algorithm is untouched: same operators, same control flow, same number of evaluations per iteration.

In [ ]:
# ----------------------------------------------------------------------------------------------
# QI-DMO: DMO dynamics (MATLAB/MEALPY-faithful) on amplitude registers
# ----------------------------------------------------------------------------------------------
def run_qidmo(inst, obj, budget, P=30, seed=0, n_baby_sitter=3, peep=2.0, mode="born_signed",
              gamma_reset=1.0, decoherence=0.0, attractor="basis", greedy_next=False, track=True, init_state=None):
    """decoherence: per-candidate depolarising strength gamma (0 = V1); gamma_reset: babysitter channel strength;
    attractor: 'basis' (alpha peeps its measured schedule) or 'register' (alpha's superposition state)."""
    rng = np.random.default_rng(seed); n, m = inst.n, inst.m
    tr = Tracker(n, m, inst); tr.purity = []
    if init_state is None:
        Psi = np.stack([uniform_state(n, m, mode) for _ in range(P)])
    else:
        Psi = init_state["Psi"].copy(); P = len(Psi)
    A = np.stack([measure(Psi[i], rng, mode) for i in range(P)]); F = np.array([obj(a) for a in A])
    C = np.zeros(P); tau = -np.inf; L = np.round(0.6 * n * n_baby_sitter)
    g = F.argmin(); gbF, gbA = F[g], A[g].copy()
    max_iter = max(1, (budget - obj.n_evals) // (3 * P))
    def snap():
        tr.snapshot(obj.n_evals, A, F, gbF); tr.purity.append(float(np.mean([purity(Psi[i], mode).mean() for i in range(P)])))
    snap()
    dec = lambda psi: depolarise(psi, decoherence, m, mode)
    for it in range(1, max_iter + 1):
        if obj.n_evals + 3 * P + n_baby_sitter > budget: break         # never exceed the evaluation budget
        CF = (1.0 - it / max_iter) ** (2.0 * it / max_iter)
        fi = np.exp(-F / F.mean()); prob = fi / fi.sum()
        # --- alpha-group foraging: candidate = alpha's peep perturbed by mongoose k
        for i in range(P):
            alpha = rng.choice(P, p=prob)
            k = rng.choice([k for k in range(P) if k not in (i, alpha)])
            phi = (peep / 2) * rng.uniform(-1, 1, (n, 1))
            E = basis_state(A[alpha], m) if attractor == "basis" else Psi[alpha]
            new = dec(project(E + phi * (E - Psi[k]), m, mode))
            a_new = measure(new, rng, mode); f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f)
            if f < F[i]: Psi[i], A[i], F[i] = new, a_new, f
            else: C[i] += 1
            if f < gbF: gbF, gbA = f, a_new.copy(); tr.gb_improvements += 1
        # --- scout group + sleeping mound
        SM = np.zeros(P)
        for i in range(P):
            k = rng.choice([k for k in range(P) if k != i])
            phi = (peep / 2) * rng.uniform(-1, 1, (n, 1))
            new = dec(project(Psi[i] + phi * (Psi[i] - Psi[k]), m, mode))
            a_new = measure(new, rng, mode); f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f)
            SM[i] = (f - F[i]) / max(f, F[i])
            if f < F[i]: Psi[i], A[i], F[i] = new, a_new, f
            else: C[i] += 1
            if f < gbF: gbF, gbA = f, a_new.copy(); tr.gb_improvements += 1
        # --- babysitter exchange = decoherence channel of strength gamma_reset (1 = full reset, classical DMO)
        for i in range(n_baby_sitter):
            if C[i] >= L:
                Psi[i] = depolarise(Psi[i], gamma_reset, m, mode); A[i] = measure(Psi[i], rng, mode); F[i] = obj(A[i]); C[i] = 0
                if F[i] < gbF: gbF, gbA = F[i], A[i].copy(); tr.gb_improvements += 1
        # --- next mongoose position (unconditional, CF-damped noise, as in the MATLAB code)
        new_tau = SM.mean()
        for i in range(P):
            phi = (peep / 2) * rng.uniform(-1, 1, (n, 1))
            if new_tau > tau: new = Psi[i] - CF * phi * rng.random() * (Psi[i] - SM[i])
            else:             new = Psi[i] + CF * phi * rng.random() * (Psi[i] - SM[i])
            tau = new_tau
            new = dec(project(new, m, mode))
            a_new = measure(new, rng, mode); f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f)
            if (not greedy_next) or f < F[i]: Psi[i], A[i], F[i] = new, a_new, f
            if f < gbF: gbF, gbA = f, a_new.copy(); tr.gb_improvements += 1
        snap()
    return {"best_f": gbF, "best_assign": gbA, "tracker": tr, "state": {"Psi": Psi}}

# ----------------------------------------------------------------------------------------------
# QI-MRFO: MRFO dynamics (MEALPY-faithful) on amplitude registers (transfer test of the mechanism)
# ----------------------------------------------------------------------------------------------
def run_qimrfo(inst, obj, budget, P=30, seed=0, S=2.0, mode="born_signed", decoherence=0.0, track=True, init_state=None, accept_equal=False,
               exchange=0.0, move_band=None):
    """accept_equal=True also accepts candidates of equal fitness (neutral drift on plateaus; V4 test).
    exchange>0 (V5, hypothesis H5 'critical exchange measurement'): with this probability a measured candidate additionally
    exchanges a task of its critical VM with a shorter task on another VM (qi_core.critical_exchange), a correlated two-register
    outcome that the product-state measurement almost never produces; the pair's registers collapse onto that outcome.
    move_band=(k_lo, k_hi) (V5, report §21 item 2 'purity-regulated gamma'): after every iteration the channel strength is
    multiplied by 1.25 if the population's expected move size n(1 - mean purity) is below k_lo tasks and divided by 1.25
    if it is above k_hi (gamma kept in [0.05/n, 8/n]); `decoherence` is then only the starting value. The trajectory
    of gamma is recorded in tracker.gamma."""
    rng = np.random.default_rng(seed); n, m = inst.n, inst.m
    tr = Tracker(n, m, inst); tr.purity = []
    if init_state is None or init_state.get("Psi") is None:        # cold start (init_state may carry only an 'elite' seed)
        Psi = np.stack([uniform_state(n, m, mode) for _ in range(P)])
    else:
        Psi = init_state["Psi"].copy(); P = len(Psi)
    A = np.stack([measure(Psi[i], rng, mode) for i in range(P)]); F = np.array([obj(a) for a in A])
    if init_state is not None and init_state.get("elite") is not None:
        # V5 elite carry-over (dynamic runs) / heuristic seed (H7, cold start): the given schedule is evaluated once
        # (charged to the budget) and replaces the worst measured individual as a (depolarised) basis state
        ea = np.asarray(init_state["elite"]); fe = obj(ea); w = int(F.argmax())
        if fe < F[w]: A[w], F[w] = ea, fe; Psi[w] = depolarise(basis_state(ea, m), decoherence, m, mode)
    g = F.argmin(); gbF, gbA = F[g], A[g].copy()
    T = max(1, (budget - obj.n_evals) // (2 * P))
    def snap():
        tr.snapshot(obj.n_evals, A, F, gbF); tr.purity.append(float(np.mean([purity(Psi[i], mode).mean() for i in range(P)])))
    snap()
    gam = [decoherence]                                   # current channel strength (mutable for the move_band controller)
    tr.gamma = [decoherence]
    dec = lambda psi: depolarise(psi, gam[0], m, mode)
    def xmeasure(new, a_new):
        # H5: correlated exchange on top of the product-state measurement; no extra random draw when exchange == 0
        if exchange <= 0 or rng.random() >= exchange: return new, a_new, False
        a_x, tt, uu = critical_exchange(inst, a_new, rng)
        rows = [tt] if uu < 0 else [tt, uu]
        return collapse_rows(new, rows, a_x[rows], gam[0], m, mode), a_x, True
    def xtrack(xm, f, f_parent):
        if xm: tr.x_cands += 1; tr.x_improving += int(f < f_parent - 1e-12)
    for t in range(1, T + 1):
        Eb = basis_state(gbA, m)
        for i in range(P):
            if rng.random() < 0.5:                       # cyclone foraging
                r1 = rng.random()
                beta = 2 * np.exp(r1 * (T - t) / T) * np.sin(2 * np.pi * r1)
                if (t + 1) / T < rng.random():           # exploration around a random schedule
                    xr = basis_state(rng.integers(0, m, n), m)
                    if i == 0: new = xr + rng.random() * (xr - Psi[i]) + beta * (xr - Psi[i])
                    else:      new = xr + rng.random() * (Psi[i - 1] - Psi[i]) + beta * (xr - Psi[i])
                else:
                    if i == 0: new = Eb + rng.random() * (Eb - Psi[i]) + beta * (Eb - Psi[i])
                    else:      new = Eb + rng.random() * (Psi[i - 1] - Psi[i]) + beta * (Eb - Psi[i])
            else:                                        # chain foraging
                r = rng.random(); alpha = 2 * r * np.sqrt(np.abs(np.log(r)))
                if i == 0: new = Psi[i] + r * (Eb - Psi[i]) + alpha * (Eb - Psi[i])
                else:      new = Psi[i] + r * (Psi[i - 1] - Psi[i]) + alpha * (Eb - Psi[i])
            new = dec(project(new, m, mode)); a_new = measure(new, rng, mode)
            new, a_new, xm = xmeasure(new, a_new)
            f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f); xtrack(xm, f, F[i])
            if f < F[i] or (accept_equal and f <= F[i]): Psi[i], A[i], F[i] = new, a_new, f
            if f < gbF: gbF, gbA = f, a_new.copy(); Eb = basis_state(gbA, m); tr.gb_improvements += 1
        for i in range(P):                               # somersault foraging
            new = dec(project(Psi[i] + S * (rng.random() * Eb - rng.random() * Psi[i]), m, mode))
            a_new = measure(new, rng, mode)
            new, a_new, xm = xmeasure(new, a_new)
            f = obj(a_new)
            if track: tr.candidate(A[i], a_new, F[i], f); xtrack(xm, f, F[i])
            if f < F[i] or (accept_equal and f <= F[i]): Psi[i], A[i], F[i] = new, a_new, f
            if f < gbF: gbF, gbA = f, a_new.copy(); Eb = basis_state(gbA, m); tr.gb_improvements += 1
        snap()
        if move_band is not None:                         # purity-regulated channel strength (report §21 item 2)
            k = n * (1.0 - tr.purity[-1])
            if k < move_band[0]: gam[0] = min(8.0 / n, max(gam[0], 0.05 / n) * 1.25)
            elif k > move_band[1]: gam[0] = max(0.05 / n, gam[0] / 1.25)
            tr.gamma.append(gam[0])
    return {"best_f": gbF, "best_assign": gbA, "tracker": tr, "state": {"Psi": Psi}}

## SECTION 10 — Validation tests
Unit tests for the objective, the encodings, the measurement law, the channel, budget accounting and determinism.

In [ ]:
def _brute_force_best(inst):
    best = np.inf
    for a in itertools.product(range(inst.m), repeat=inst.n):
        best = min(best, Objective(inst)._raw(np.array(a))[0])
    return best

tests = {}
_t = make_instance(6, 3, seed=5)
# 1. objective vs brute force + lower bound validity
_bf = _brute_force_best(_t); tests["brute_force_optimum >= LB"] = _bf >= _t.lower_bound() - 1e-9
_ga = run_ga(_t, Objective(_t), 3000, P=20, seed=0); tests["GA on 6x3 finds the brute-force optimum"] = abs(_ga["best_f"] - _bf) < 1e-9
# 2. decoder bounds
tests["decode stays in [0, m-1]"] = decode(np.array([-1.0, 0.0, 2.999, 3.0, 99.0]), 3).tolist() == [0, 0, 2, 2, 2]
# 3. measurement follows the Born rule (chi-square goodness of fit on 20000 draws)
_psi = renormalise(np.array([[0.1, 0.5, 0.3, 0.8]]), 4); _p = born_probs(_psi)[0]
_draws = np.array([measure(_psi, np.random.default_rng(k), "born_signed")[0] for k in range(20000)])
_counts = np.bincount(_draws, minlength=4); _chi2, _pv = stats.chisquare(_counts, 20000 * _p)
tests["measurement ~ Born rule (chi2 p>0.001)"] = _pv > 0.001
# 4. channel keeps normalisation and lowers purity monotonically
_E = basis_state(np.array([1, 2, 0]), 4); _pur = [purity(depolarise(_E, g, 4, "born_signed"), "born_signed").mean() for g in [0, 0.1, 0.5, 1.0]]
tests["channel preserves normalisation"] = np.allclose((depolarise(_E, 0.3, 4, "born_signed") ** 2).sum(1), 1)
tests["purity decreases with gamma, uniform at gamma=1"] = all(np.diff(_pur) < 0) and abs(_pur[-1] - 0.25) < 1e-12
# 5. budget accounting: no optimizer exceeds its evaluation budget
for _name, _fn in {"PSO": run_pso, "DMO": run_dmo, "MRFO": run_mrfo, "GA": run_ga, "QI-MRFO": run_qimrfo, "QI-DMO": run_qidmo}.items():
    _o = Objective(_t); _fn(_t, _o, 700, P=10, seed=1); tests[f"{_name} respects the budget"] = _o.n_evals <= 700
# 6. determinism
_r1 = run_qimrfo(_t, Objective(_t), 600, P=10, seed=7, decoherence=0.1); _r2 = run_qimrfo(_t, Objective(_t), 600, P=10, seed=7, decoherence=0.1)
tests["same seed -> same result"] = _r1["best_f"] == _r2["best_f"] and np.array_equal(_r1["best_assign"], _r2["best_assign"])
# 7. gamma = 1 on every candidate makes QI-MRFO a random sampler (purity stays 1/m)
_r3 = run_qimrfo(_t, Objective(_t), 600, P=10, seed=7, decoherence=1.0); tests["gamma=1 -> uniform registers (purity 1/m)"] = abs(_r3["tracker"].purity[-1] - 1 / _t.m) < 1e-9
# 8. valid schedules from all modes
for _mode in ["born_signed", "born_abs", "linear"]:
    _r = run_qimrfo(_t, Objective(_t), 400, P=8, seed=2, mode=_mode, decoherence=0.05); tests[f"valid schedule ({_mode})"] = _r["best_assign"].min() >= 0 and _r["best_assign"].max() < _t.m
# 9. the linear twin's basis-state and measurement agree with the Born version for a one-hot register
tests["one-hot register measures deterministically"] = np.array_equal(measure(basis_state(np.array([2, 0, 1]), 3), np.random.default_rng(0), "born_signed"), np.array([2, 0, 1]))
for k, v in tests.items(): print(("PASS" if v else "FAIL"), "-", k)
assert all(tests.values()), "validation failed"

### Experiment harness
`run_suite` runs every (algorithm, instance, seed) combination with equal evaluation budgets and returns one record per run
(best value, gap to LB, runtime, wasted/neutral/improving fractions, mean move size, final diversity, final purity, curves).
Runs are parallelised with `fork` when available and silently fall back to a sequential loop otherwise.

In [ ]:
def _heur(fn):
    def _run(inst, obj, budget, seed):
        a = fn(inst); f = obj(a); tr = Tracker(inst.n, inst.m); tr.snapshot(obj.n_evals, [a], [f], f)
        return {"best_f": f, "best_assign": a, "tracker": tr}
    return _run

def _mk(fn, **kw):
    def _run(inst, obj, budget, seed):
        k = {key: (val(inst) if callable(val) else val) for key, val in kw.items()}
        return fn(inst, obj, budget, seed=seed, **k)
    return _run

REGISTRY = {
    "Max-Min": _heur(max_min), "Min-Min": _heur(min_min),
    "Random": _mk(run_random, P=CFG["P"]),
    "PSO": _mk(run_pso, P=CFG["P"]), "DMO": _mk(run_dmo, P=CFG["P"]), "MRFO": _mk(run_mrfo, P=CFG["P"]), "GA": _mk(run_ga, P=CFG["P"]),
    "QI-MRFO": _mk(run_qimrfo, P=CFG["P"], decoherence=lambda inst: CFG["gamma_c_mrfo"] / inst.n),
    "QI-DMO": _mk(run_qidmo, P=CFG["P"], decoherence=lambda inst: CFG["gamma_c_dmo"] / inst.n),
    "P-MRFO (linear twin)": _mk(run_qimrfo, P=CFG["P"], mode="linear", decoherence=lambda inst: CFG["gamma_c_mrfo"] / inst.n),
    "P-DMO (linear twin)": _mk(run_qidmo, P=CFG["P"], mode="linear", decoherence=lambda inst: CFG["gamma_c_dmo"] / inst.n),
    "QI-MRFO no-decoherence": _mk(run_qimrfo, P=CFG["P"], decoherence=0.0),
    "QI-DMO no-decoherence": _mk(run_qidmo, P=CFG["P"], decoherence=0.0),
    "QI-MRFO unsigned": _mk(run_qimrfo, P=CFG["P"], mode="born_abs", decoherence=lambda inst: CFG["gamma_c_mrfo"] / inst.n),
    "QI-DMO register-attractor": _mk(run_qidmo, P=CFG["P"], attractor="register", decoherence=lambda inst: CFG["gamma_c_dmo"] / inst.n),
    "DMO greedy-next": _mk(run_dmo, P=CFG["P"], greedy_next=True),
    "QI-DMO greedy-next": _mk(run_qidmo, P=CFG["P"], greedy_next=True, decoherence=lambda inst: CFG["gamma_c_dmo"] / inst.n),
}

def instance_from_spec(spec, inst_seed=None):
    n, m, dist, het = spec
    return make_instance(n, m, seed=CFG["inst_seed"] if inst_seed is None else inst_seed, task_dist=dist, hetero=het)

def run_one(job):
    algo, spec, seed, budget, objective, inst_seed = job
    inst = instance_from_spec(spec, inst_seed); obj = Objective(inst, kind=objective)
    t0 = time.time(); r = REGISTRY[algo](inst, obj, budget, seed); rt = time.time() - t0
    tr = r["tracker"]; s = tr.summary(); lb = inst.lower_bound(); det = obj.details(r["best_assign"])
    rec = {"algo": algo, "instance": f"n{spec[0]} m{spec[1]} {spec[2]} {spec[3]}", "n": spec[0], "m": spec[1], "seed": seed, "budget": budget,
           "best": float(r["best_f"]), "makespan": det["makespan"], "energy_Wh": det["energy_Wh"], "lb": lb,
           "gap": (det["makespan"] - lb) / lb if objective == "makespan" else np.nan, "runtime_s": rt, "evals": obj.n_evals,
           "wasted": s["wasted_frac"], "neutral": s["neutral_frac"], "improving": s["improving_frac"], "move_size": s["mean_move_size"],
           "gb_impr_per_1k": s["gb_impr_per_1k"], "div_end": tr.div_ham[-1], "purity_end": (tr.purity[-1] if hasattr(tr, "purity") else np.nan),
           "curve_evals": np.array(tr.evals), "curve_best": np.array(tr.best), "curve_div": np.array(tr.div_ham),
           "curve_purity": (np.array(tr.purity) if hasattr(tr, "purity") else None),
           "move_sizes_early": float(np.mean(tr.move_sizes[: len(tr.move_sizes) // 2])) if tr.move_sizes else np.nan,
           "move_sizes_late": float(np.mean(tr.move_sizes[len(tr.move_sizes) // 2:])) if tr.move_sizes else np.nan}
    # V5 diagnostics (Section 19): tighter preemptive LB, global duplicates, critical-VM touches, exchange moves,
    # stagnation time, and the improving relocations / swaps still available at the returned schedule
    s5 = tr.summary_v5(); lb2 = inst.lower_bound_pmtn(); rel, swp, _ = count_improving_moves(inst, r["best_assign"])
    rec.update({"lb2": lb2, "gap2": (det["makespan"] - lb2) / lb2 if objective == "makespan" else np.nan,
                "dup_global": s5["dup_global_frac"], "touch_crit": s5["touch_crit_frac"], "x_frac": s5["x_frac"], "x_success": s5["x_success"],
                "late_improving": s5["late_improving_frac"], "last_gb_impr_frac": s5["last_gb_impr_frac"], "end_impr_reloc": rel, "end_impr_swap": swp})
    return rec

def run_suite(algos, specs, seeds, budget, objective=None, inst_seed=None, label=""):
    objective = objective or CFG["objective"]
    jobs = [(a, s, sd, budget, objective, inst_seed) for s in specs for a in algos for sd in seeds]
    t0 = time.time(); recs = None
    if CFG["parallel"] and len(jobs) > 1:
        try:
            import multiprocessing as mp
            if mp.get_start_method(allow_none=True) in (None, "fork") and hasattr(os, "fork"):
                from concurrent.futures import ProcessPoolExecutor
                with ProcessPoolExecutor(max_workers=min(os.cpu_count() or 1, len(jobs)), mp_context=mp.get_context("fork")) as ex:
                    recs = list(ex.map(run_one, jobs, chunksize=1))
        except Exception as e:
            print("parallel execution unavailable ->", type(e).__name__, "; running sequentially"); recs = None
    if recs is None:
        recs = []
        for k, j in enumerate(jobs):
            recs.append(run_one(j))
            if (k + 1) % 20 == 0: print(f"  {k + 1}/{len(jobs)} runs done ({time.time() - t0:.0f}s)")
    df = pd.DataFrame(recs)
    print(f"{label}: {len(jobs)} runs in {time.time() - t0:.0f}s")
    return df

def summarize(df, by=("instance", "algo")):
    g = df.groupby(list(by))
    out = g.agg(best_mean=("best", "mean"), best_sd=("best", "std"), best_median=("best", "median"), best_min=("best", "min"),
                gap_mean=("gap", "mean"), runtime_s=("runtime_s", "mean"), wasted=("wasted", "mean"), move_size=("move_size", "mean"),
                div_end=("div_end", "mean"), purity_end=("purity_end", "mean"), gb_impr_per_1k=("gb_impr_per_1k", "mean")).reset_index()
    out["gap_mean"] = (100 * out["gap_mean"]).round(2)
    return out.round(3)

# fixed categorical palette (colour follows the entity, never the rank); thin marks, hairline grid
PALETTE = {"QI-MRFO": "#2a78d6", "MRFO": "#eb6834", "GA": "#1baf7a", "QI-DMO": "#eda100", "DMO": "#e87ba4", "PSO": "#008300",
           "P-MRFO (linear twin)": "#4a3aa7", "Random": "#e34948", "P-DMO (linear twin)": "#4a3aa7", "Max-Min": "#898781"}
def style_axes(ax, title="", xlabel="", ylabel=""):
    ax.set_title(title, fontsize=10, loc="left", color="#0b0b0b"); ax.set_xlabel(xlabel, fontsize=9, color="#52514e"); ax.set_ylabel(ylabel, fontsize=9, color="#52514e")
    ax.grid(True, color="#e1e0d9", linewidth=0.6); ax.set_axisbelow(True)
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
    for sp in ["left", "bottom"]: ax.spines[sp].set_color("#c3c2b7")
    ax.tick_params(colors="#898781", labelsize=8)
def color_of(name): return PALETTE.get(name, "#898781")
print("harness ready;", len(REGISTRY), "algorithms registered")

## SECTION 11 — Small smoke experiment
A 10-task / 3-VM instance, two seeds, 2000 evaluations: every algorithm must run end-to-end and return a valid schedule.

In [ ]:
_smoke_specs = [(10, 3, "uniform", "high")]
smoke = run_suite(["Max-Min", "Random", "PSO", "DMO", "MRFO", "GA", "QI-MRFO", "QI-DMO"], _smoke_specs, [0, 1], 2000, label="smoke")
summarize(smoke)[["instance", "algo", "best_mean", "gap_mean", "runtime_s", "wasted", "move_size", "purity_end"]]

## SECTION 12 — Baseline experiment (equal budget, common instances, multiple seeds)

**Prediction stated before running (from the pilot):** classical DMO/MRFO degrade towards random search as $n$ grows; the
quantum-inspired versions remove that pathology (move size becomes controllable), with the gain growing with $n$; QI-MRFO should
match the discrete GA; the linear twin should capture most of the gain if the benefit is representational. Effect sizes are only
claimed after the run below. The Max-Min list heuristic is a strong, cheap reference: a metaheuristic that cannot beat it on a static
makespan batch has no practical case there.

In [ ]:
BASE_ALGOS = ["Max-Min", "Min-Min", "Random", "PSO", "DMO", "MRFO", "GA", "QI-MRFO", "QI-DMO", "P-MRFO (linear twin)"]
baseline = run_suite(BASE_ALGOS, CFG["instances"], CFG["seeds"], CFG["budget"], label="baseline")
baseline.drop(columns=[c for c in baseline.columns if c.startswith("curve_")]).to_csv(os.path.join(CFG["results_dir"], f"baseline_{MODE}.csv"), index=False)
pickle.dump(baseline, open(os.path.join(CFG["results_dir"], f"baseline_{MODE}.pkl"), "wb"))
base_sum = summarize(baseline)
base_sum.to_csv(os.path.join(CFG["results_dir"], f"baseline_summary_{MODE}.csv"), index=False)
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200)
base_sum[["instance", "algo", "best_mean", "best_sd", "best_median", "gap_mean", "runtime_s", "wasted", "move_size", "div_end", "purity_end", "gb_impr_per_1k"]]

In [ ]:
# gap-to-lower-bound table: instances x algorithms (mean over seeds, %)
pivot_gap = baseline.pivot_table(index="instance", columns="algo", values="gap", aggfunc="mean") * 100
pivot_gap = pivot_gap[[a for a in BASE_ALGOS if a in pivot_gap.columns]].round(2)
print("Mean gap to the lower bound (%):"); pivot_gap

## SECTION 13 — Convergence plots
Median best-so-far gap (inter-quartile band) versus evaluations, population diversity, register purity, and the move-size
diagnostic (early vs late half of the run). One axis per panel; colours follow the algorithm identity throughout the notebook.

In [ ]:
def curve_grid(df, algo, instance, key="curve_best", npts=120):
    sub = df[(df.algo == algo) & (df.instance == instance)]
    if len(sub) == 0: return None, None
    xmax = max(r.curve_evals[-1] for r in sub.itertuples())
    grid = np.linspace(0, xmax, npts); ys = []
    for r in sub.itertuples():
        y = getattr(r, key)
        if y is None: return None, None
        ys.append(np.interp(grid, r.curve_evals, y))
    return grid, np.array(ys)

PLOT_ALGOS = ["MRFO", "QI-MRFO", "GA", "DMO", "QI-DMO", "PSO"]
insts = list(dict.fromkeys(baseline.instance))
fig, axes = plt.subplots(1, len(insts), figsize=(4.2 * len(insts), 3.4), squeeze=False)
for ax, inst_name in zip(axes[0], insts):
    lb = baseline[baseline.instance == inst_name].lb.iloc[0]
    for algo in PLOT_ALGOS:
        grid, ys = curve_grid(baseline, algo, inst_name)
        if grid is None: continue
        gap = (ys - lb) / lb * 100
        ax.plot(grid, np.median(gap, 0), color=color_of(algo), linewidth=1.8, label=algo)
        ax.fill_between(grid, np.percentile(gap, 25, 0), np.percentile(gap, 75, 0), color=color_of(algo), alpha=0.12, linewidth=0)
    ax.set_yscale("log"); style_axes(ax, inst_name, "evaluations", "gap to LB (%)  [log]")
axes[0][0].legend(fontsize=8, frameon=False)
fig.suptitle("Convergence (median, IQR band)", x=0.01, ha="left", fontsize=11); fig.tight_layout(); plt.savefig(os.path.join(CFG["results_dir"], f"fig_convergence_{MODE}.png"), dpi=130); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(insts), figsize=(4.2 * len(insts), 3.2), squeeze=False)
for ax, inst_name in zip(axes[0], insts):
    for algo in PLOT_ALGOS:
        grid, ys = curve_grid(baseline, algo, inst_name, key="curve_div")
        if grid is None: continue
        ax.plot(grid, np.median(ys, 0), color=color_of(algo), linewidth=1.8, label=algo)
    style_axes(ax, inst_name, "evaluations", "diversity (mean Hamming / n)")
axes[0][0].legend(fontsize=8, frameon=False)
fig.suptitle("Population diversity", x=0.01, ha="left", fontsize=11); fig.tight_layout(); plt.savefig(os.path.join(CFG["results_dir"], f"fig_diversity_{MODE}.png"), dpi=130); plt.show()

fig, axes = plt.subplots(1, len(insts), figsize=(4.2 * len(insts), 3.2), squeeze=False)
for ax, inst_name in zip(axes[0], insts):
    for algo in ["QI-MRFO", "QI-DMO", "P-MRFO (linear twin)"]:
        grid, ys = curve_grid(baseline, algo, inst_name, key="curve_purity")
        if grid is None: continue
        ax.plot(grid, np.median(ys, 0), color=color_of(algo), linewidth=1.8, label=algo)
    style_axes(ax, inst_name, "evaluations", "mean register purity")
axes[0][0].legend(fontsize=8, frameon=False)
fig.suptitle("Register purity (1 = collapsed, 1/m = uniform)", x=0.01, ha="left", fontsize=11); fig.tight_layout(); plt.savefig(os.path.join(CFG["results_dir"], f"fig_purity_{MODE}.png"), dpi=130); plt.show()

In [ ]:
# move-size diagnostic: how many tasks does a candidate change, early vs late in the run (largest instance)
big = max(insts, key=lambda s: int(s.split()[0][1:]))
ms = baseline[baseline.instance == big].groupby("algo")[["move_sizes_early", "move_sizes_late"]].mean().reindex([a for a in PLOT_ALGOS + ["P-MRFO (linear twin)"] if a in set(baseline.algo)])
fig, ax = plt.subplots(figsize=(7, 3.2)); x = np.arange(len(ms)); w = 0.38
ax.bar(x - w / 2, ms.move_sizes_early, w, color=[color_of(a) for a in ms.index], alpha=0.55, label="first half of run", linewidth=0)
ax.bar(x + w / 2, ms.move_sizes_late, w, color=[color_of(a) for a in ms.index], label="second half of run", linewidth=0)
ax.set_xticks(x); ax.set_xticklabels(ms.index, fontsize=8, rotation=15); style_axes(ax, f"Mean number of tasks changed per candidate — {big}", "", "tasks changed")
ax.legend(fontsize=8, frameon=False); fig.tight_layout(); plt.savefig(os.path.join(CFG["results_dir"], f"fig_movesize_{MODE}.png"), dpi=130); plt.show()
ms.round(2)

## SECTION 14 — Statistical analysis
Paired comparisons by seed (Wilcoxon signed-rank; Holm correction across instances), non-parametric effect sizes
(Cliff's $\delta$ and Vargha–Delaney $A_{12}$), bootstrap 95 % confidence intervals of the mean gap difference, and a Friedman test over
algorithms. With few seeds (smoke/fast modes) p-values are reported but are not expected to reach significance; the `full` mode has 30 seeds.

In [ ]:
def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y); gt = (x[:, None] > y[None, :]).mean(); lt = (x[:, None] < y[None, :]).mean(); return gt - lt
def a12(x, y):
    x, y = np.asarray(x), np.asarray(y); return ((x[:, None] < y[None, :]).mean() + 0.5 * (x[:, None] == y[None, :]).mean())   # P(x < y): >0.5 means x is better (minimisation)
def boot_ci(d, B=4000, seed=0):
    d = np.asarray(d); rng = np.random.default_rng(seed); bs = [rng.choice(d, len(d), replace=True).mean() for _ in range(B)]; return np.percentile(bs, [2.5, 97.5])
def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); adj = np.empty_like(p); m = len(p); running = 0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx]); adj[idx] = min(1.0, running)
    return adj

PAIRS = [("QI-MRFO", "MRFO"), ("QI-MRFO", "GA"), ("QI-MRFO", "P-MRFO (linear twin)"), ("QI-MRFO", "Max-Min"), ("QI-DMO", "DMO"), ("QI-DMO", "GA"), ("QI-MRFO", "QI-DMO")]
rows = []
for a, b in PAIRS:
    for inst_name in insts:
        xa = baseline[(baseline.algo == a) & (baseline.instance == inst_name)].sort_values("seed").gap.values
        xb = baseline[(baseline.algo == b) & (baseline.instance == inst_name)].sort_values("seed").gap.values
        if len(xa) == 0 or len(xb) == 0: continue
        if len(xa) == len(xb) and len(xa) >= 2 and np.any(xa != xb):
            p = stats.wilcoxon(xa, xb, zero_method="zsplit").pvalue if len(xa) >= 5 else stats.ttest_rel(xa, xb).pvalue
        else: p = np.nan
        d = 100 * (xa - xb); lo, hi = boot_ci(d) if len(d) >= 2 else (np.nan, np.nan)
        rows.append({"A": a, "B": b, "instance": inst_name, "gap A %": 100 * xa.mean(), "gap B %": 100 * xb.mean(), "diff (A-B) pp": d.mean(), "CI95 lo": lo, "CI95 hi": hi,
                     "p": p, "cliffs_delta": cliffs_delta(xa, xb), "A12 (P[A better])": a12(xa, xb)})
stat = pd.DataFrame(rows)
stat["p_holm"] = np.nan
for (a, b), g in stat.groupby(["A", "B"]):
    mask = g.p.notna()
    if mask.any(): stat.loc[g.index[mask], "p_holm"] = holm(g.p[mask].values)
stat.to_csv(os.path.join(CFG["results_dir"], f"stats_{MODE}.csv"), index=False)
stat.round(4)

In [ ]:
# average ranks across instances (lower is better) + Friedman test on per-(instance, seed) blocks
wide = baseline.pivot_table(index=["instance", "seed"], columns="algo", values="best")
ranks = wide.rank(axis=1, method="average")
print("mean rank over all (instance, seed) blocks:"); print(ranks.mean().sort_values().round(2).to_string())
if wide.shape[0] >= 3 and wide.shape[1] >= 3:
    fr = stats.friedmanchisquare(*[wide[c].values for c in wide.columns]); print(f"\nFriedman chi2 = {fr.statistic:.2f}, p = {fr.pvalue:.2e}")

## SECTION 15 — Ablation: did the quantum-inspired component cause the effect?
Four ladders on the primary host (and the same on DMO):
1. **MRFO** (classical, floor encoding) → 2. **QI-MRFO no-decoherence** (registers + measurement only) → 3. **QI-MRFO** (full) — isolates the representation and the channel.
4. **QI-MRFO unsigned** removes the sign (no interference possible). 5. **P-MRFO (linear twin)** replaces the Born rule by linear probability mixing: the classical equivalent.
6. **QI-DMO register-attractor** uses the alpha's superposition instead of its measured schedule as attractor (negative control for collapse-conditioned attraction).
7. **DMO greedy-next / QI-DMO greedy-next** test whether DMO's unconditional 'next position' move explains the QI-DMO vs QI-MRFO difference.

In [ ]:
ABL_SPECS = CFG["instances"][:4] if MODE == "full" else CFG["instances"][:2] + CFG["instances"][2:3]
ABL_ALGOS = ["MRFO", "QI-MRFO no-decoherence", "QI-MRFO", "QI-MRFO unsigned", "P-MRFO (linear twin)",
             "DMO", "QI-DMO no-decoherence", "QI-DMO", "P-DMO (linear twin)", "QI-DMO register-attractor", "DMO greedy-next", "QI-DMO greedy-next", "GA"]
ablation = run_suite(ABL_ALGOS, ABL_SPECS, CFG["seeds"], CFG["budget"], label="ablation")
ablation.drop(columns=[c for c in ablation.columns if c.startswith("curve_")]).to_csv(os.path.join(CFG["results_dir"], f"ablation_{MODE}.csv"), index=False)
abl_sum = summarize(ablation)
abl_pivot = ablation.pivot_table(index="algo", columns="instance", values="gap", aggfunc="mean").reindex(ABL_ALGOS) * 100
print("Ablation: mean gap to LB (%)"); abl_pivot.round(2)

In [ ]:
abl_sum[["instance", "algo", "best_mean", "best_sd", "gap_mean", "wasted", "move_size", "div_end", "purity_end"]]

## SECTION 16 — Parameter sensitivity
Decoherence strength $\gamma = c/n$ (dose–response), population size $P$ and the somersault factor $S$ for QI-MRFO; $\gamma$ for QI-DMO.
The pilot chose $c=1$ (MRFO) and $c=0.25$ (DMO); this section shows how flat or sharp the optimum is.

In [ ]:
SENS_SPECS = [CFG["instances"][1], CFG["instances"][2], CFG["instances"][4]] if MODE == "full" else CFG["instances"][1:3]
for c in [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]:
    REGISTRY[f"QI-MRFO c={c}"] = _mk(run_qimrfo, P=CFG["P"], decoherence=(lambda inst, c=c: c / inst.n))
    REGISTRY[f"QI-DMO c={c}"] = _mk(run_qidmo, P=CFG["P"], decoherence=(lambda inst, c=c: c / inst.n))
for Pp in [15, 60]:
    REGISTRY[f"QI-MRFO P={Pp}"] = _mk(run_qimrfo, P=Pp, decoherence=lambda inst: CFG["gamma_c_mrfo"] / inst.n)
for Ss in [1.0, 3.0]:
    REGISTRY[f"QI-MRFO S={Ss}"] = _mk(run_qimrfo, P=CFG["P"], S=Ss, decoherence=lambda inst: CFG["gamma_c_mrfo"] / inst.n)
SENS_ALGOS = [f"QI-MRFO c={c}" for c in [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]] + [f"QI-DMO c={c}" for c in [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]] + ["QI-MRFO P=15", "QI-MRFO P=60", "QI-MRFO S=1.0", "QI-MRFO S=3.0"]
sens = run_suite(SENS_ALGOS, SENS_SPECS, CFG["seeds"], CFG["budget"], label="sensitivity")
sens.drop(columns=[c for c in sens.columns if c.startswith("curve_")]).to_csv(os.path.join(CFG["results_dir"], f"sensitivity_{MODE}.csv"), index=False)
sens_pivot = sens.pivot_table(index="algo", columns="instance", values="gap", aggfunc="mean").reindex(SENS_ALGOS) * 100
sens_w = sens.pivot_table(index="algo", columns="instance", values="wasted", aggfunc="mean").reindex(SENS_ALGOS)
print("Sensitivity: mean gap to LB (%)"); display(sens_pivot.round(2)); print("wasted candidate fraction"); sens_w.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
cs = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
for ax, host, col in zip(axes, ["QI-MRFO", "QI-DMO"], ["#2a78d6", "#eda100"]):
    for k, inst_name in enumerate(list(dict.fromkeys(sens.instance))):
        y = [sens[(sens.algo == f"{host} c={c}") & (sens.instance == inst_name)].gap.mean() * 100 for c in cs]
        ax.plot(range(len(cs)), y, marker="o", markersize=5, linewidth=1.8, color=col, alpha=1 - 0.35 * k, label=inst_name)
    ax.set_xticks(range(len(cs))); ax.set_xticklabels([str(c) for c in cs]); style_axes(ax, f"{host}: decoherence dose-response", "c  (gamma = c / n)", "gap to LB (%)"); ax.legend(fontsize=7, frameon=False)
fig.tight_layout(); plt.savefig(os.path.join(CFG["results_dir"], f"fig_sensitivity_{MODE}.png"), dpi=130); plt.show()

## SECTION 17 — Failure-case (adversarial) experiments
Regimes where the mechanism is *expected not to help* or to lose:
* **17a** tiny search space ($n=10, m=3$) — random search may suffice;
* **17b** two VMs ($m=2$) — the floor encoding is almost lossless, so the representational advantage should vanish;
* **17c** identical tasks on homogeneous VMs — a trivial landscape, exploitation should dominate;
* **17d** the smooth `makespan_energy` objective — additive, no plateaus; heuristics are not designed for it;
* **17e** many VMs ($n=40, m=20$) — large registers, possible over-fast collapse;
* **17f** the bimodal (tall-barrier) instance against the Max-Min heuristic;
* **17g** runtime and memory overhead of the register representation;
* **17h** dynamic workloads (task churn, VM failure, VM addition, speed drift) — the cloud-specific extension: restart vs continue vs decoherence shock vs GA hypermutation.

In [ ]:
FAIL_ALGOS = ["Max-Min", "Random", "MRFO", "QI-MRFO", "GA", "DMO", "QI-DMO"]
fail_cases = {"17a tiny": (10, 3, "uniform", "high"), "17b two VMs": (40, 2, "uniform", "high"), "17c identical tasks, homogeneous VMs": (40, 8, "identical", "none"),
              "17e many VMs": (40, 20, "uniform", "high"), "17f bimodal tall barriers": (50, 10, "bimodal", "high")}
fail_frames = []
for label, spec in fail_cases.items():
    df = run_suite(FAIL_ALGOS, [spec], CFG["seeds"], min(CFG["budget"], 10000), label=label); df["case"] = label; fail_frames.append(df)
# 17d smooth objective (makespan + energy); the 'best' column is the combined objective, 'makespan'/'energy_Wh' are the components
df = run_suite(FAIL_ALGOS, [CFG["instances"][min(2, len(CFG["instances"]) - 1)]], CFG["seeds"], min(CFG["budget"], 10000), objective="makespan_energy", label="17d makespan_energy"); df["case"] = "17d makespan+energy"; fail_frames.append(df)
# 17c follow-up (V4 of the research loop): the identical-task plateau exposed that QI-MRFO's strict greedy acceptance blocks
# neutral drift. Test the classical fix — accept equal-fitness candidates — on the plateau instance and on a normal one.
REGISTRY["QI-MRFO neutral-accept"] = _mk(run_qimrfo, P=CFG["P"], accept_equal=True, decoherence=lambda inst: CFG["gamma_c_mrfo"] / inst.n)
for label, spec in {"17c identical tasks, homogeneous VMs": (40, 8, "identical", "none"), "17c-control uniform n100": CFG["instances"][min(2, len(CFG["instances"]) - 1)]}.items():
    df = run_suite(["QI-MRFO", "QI-MRFO neutral-accept"], [spec], CFG["seeds"], min(CFG["budget"], 10000), label=label + " (neutral acceptance)"); df["case"] = label + " / neutral-accept test"; fail_frames.append(df)
fail = pd.concat(fail_frames, ignore_index=True)
fail.drop(columns=[c for c in fail.columns if c.startswith("curve_")]).to_csv(os.path.join(CFG["results_dir"], f"failure_cases_{MODE}.csv"), index=False)
fail_tab = fail.groupby(["case", "algo"]).agg(best_mean=("best", "mean"), best_sd=("best", "std"), makespan=("makespan", "mean"), energy_Wh=("energy_Wh", "mean"), gap_pct=("gap", lambda g: 100 * g.mean()), wasted=("wasted", "mean"), move_size=("move_size", "mean")).round(3)
fail_tab

In [ ]:
# 17g runtime / memory overhead of the register representation
ov_spec = (100, 10, "uniform", "high"); ov_inst = instance_from_spec(ov_spec)
ov = {}
for name in ["MRFO", "QI-MRFO", "DMO", "QI-DMO", "GA"]:
    t0 = time.time(); r = REGISTRY[name](ov_inst, Objective(ov_inst), 6000, 0); ov[name] = {"s_per_1k_evals": (time.time() - t0) / 6.0,
          "state_bytes": (CFG["P"] * ov_inst.n * ov_inst.m * 8 if name.startswith("QI") else CFG["P"] * ov_inst.n * 8)}
ov = pd.DataFrame(ov).T; ov["overhead_vs_classical"] = [ov.loc[k, "s_per_1k_evals"] / ov.loc[k[3:], "s_per_1k_evals"] if k.startswith("QI") else 1.0 for k in ov.index]
print(f"instance n={ov_inst.n}, m={ov_inst.m}, P={CFG['P']}"); ov.round(4)

### 17h — Dynamic workloads (cloud-specific extension pilot)
Hypothesis: after a workload change, a swarm that has collapsed must re-diversify; full restart discards all information, naive
continuation keeps a stale, collapsed population. The depolarising channel gives a *controlled forgetting* knob $\gamma_{shock}$; the
classical equivalent for the GA is hypermutation. The register representation also has clean rules for structural change:
VM removal = column deletion (projective measurement), VM addition = a new column with a uniform share, new tasks = uniform registers.

In [ ]:
def make_dynamic_sequence(n, m, seed, K=6, change="churn", rho=0.2, task_dist="uniform", hetero="high"):
    """Returns [(instance, change_info)]; change_info of epoch 0 is None."""
    rng = np.random.default_rng(seed + 1000)
    inst = make_instance(n, m, seed, task_dist, hetero)
    seq = [(inst, None)]
    for k in range(K):
        ctype = change
        if change == "mixed":                       # V5: a random event per epoch (VM failure only while > 3 VMs remain)
            ctype = str(rng.choice(["churn", "drift", "vm_fail", "vm_add"]))
            if ctype == "vm_fail" and inst.m <= 3: ctype = "vm_add"
        if ctype == "churn":                        # rho of the tasks complete and are replaced by new arrivals
            idx = rng.choice(inst.n, max(1, int(rho * inst.n)), replace=False)
            L = inst.task_len.copy(); L[idx] = sample_task_lengths(rng, len(idx), task_dist)
            inst = inst.copy_with(task_len=L); info = {"type": "churn", "idx": idx}
        elif ctype == "vm_fail":                   # one VM disappears (spot instance reclaimed / failure)
            j = int(rng.integers(inst.m))
            inst = CloudInstance(inst.task_len, np.delete(inst.vm_mips, j), np.delete(inst.vm_p_idle, j), np.delete(inst.vm_p_max, j))
            info = {"type": "vm_fail", "j": j}
        elif ctype == "vm_add":                    # a VM is added (scale-out)
            S = sample_vm_speeds(rng, 1, hetero); pm_ = 100.0 + 0.1 * S
            inst = CloudInstance(inst.task_len, np.append(inst.vm_mips, S), np.append(inst.vm_p_idle, 0.6 * pm_), np.append(inst.vm_p_max, pm_))
            info = {"type": "vm_add"}
        elif ctype == "drift":                     # VM speeds drift (contention / noisy neighbours)
            inst = inst.copy_with(vm_mips=inst.vm_mips * rng.lognormal(0, 0.3, inst.m)); info = {"type": "drift"}
        else:
            raise ValueError(change)
        seq.append((inst, info))
    return seq

# --- state adaptation to structural changes ---------------------------------------------------------------
def adapt_register_state(state, info, mode, rng, struct=True):
    """Shape changes (VM removed/added) must always be applied; `struct` additionally resets the registers of
    replaced (new) tasks to the uniform superposition instead of keeping the old task's register."""
    if info is None: return state
    if info["type"] == "vm_fail": return remove_vm_state(state, info["j"], mode)
    if info["type"] == "vm_add": return add_vm_state(state, mode)
    if info["type"] == "churn" and struct: return add_tasks_state(state, info["idx"], mode)
    return state

def adapt_position_state(state, info, m_new, rng):
    """Continuous floor-encoded positions under structural change (the standard swarm encoding has no clean rule)."""
    if info is None: return state
    X = state["X"].copy()
    if info["type"] == "vm_fail":
        j = info["j"]
        on_j = (X >= j) & (X < j + 1)
        X[X >= j + 1] -= 1
        X[on_j] = rng.uniform(0, m_new, on_j.sum())
    new = dict(state); new["X"] = X
    return new

def adapt_ga_state(state, info, m_new, rng, inst=None, repair="random"):
    if info is None: return state
    A = state["A"].copy()
    if repair == "incremental" and info["type"] in ("churn", "vm_fail"):
        # V5, H12: every carried schedule keeps its persistent tasks and gets the new / orphaned tasks placed longest
        # first on the VM that finishes them earliest (incremental_list_schedule)
        return {"A": np.stack([incremental_list_schedule(a, info, inst, rng) for a in A])}
    if info["type"] == "vm_fail":
        if repair == "greedy":
            return {"A": np.stack([repair_schedule(a, info, inst, rng, "greedy") for a in A])}
        j = info["j"]; A[A > j] -= 1; bad = A == j; A[bad] = rng.integers(0, m_new, bad.sum())
    return {"A": A}

def map_to_new_vms(a, info):
    """Map a schedule of the previous epoch onto the new VM indexing. Returns (mapped, forced) where forced marks the
    tasks whose VM disappeared (mapped value -1)."""
    a = np.asarray(a).copy(); forced = np.zeros(len(a), bool)
    if info is not None and info["type"] == "vm_fail":
        j = info["j"]; forced = a == j; a[a > j] -= 1; a[forced] = -1
    return a, forced

def repair_schedule(a, info, inst_new, rng, how="random"):
    """Make the previous epoch's schedule valid for the new instance. Only a VM failure invalidates it: the tasks of
    the failed VM go to uniformly random VMs ('random', as adapt_ga_state) or, longest first, to the VM that
    finishes them earliest ('greedy', list scheduling on top of the surviving loads). With 'random' and 'greedy' the
    NEW tasks of a churn event stay on the VM of the departed task whose index they took (slot inheritance).
    'incremental' (V5, H12) also places the new tasks longest first on the VM that finishes them earliest, i.e. it
    returns incremental_list_schedule (identical to 'greedy' for every event except churn)."""
    if how == "incremental": return incremental_list_schedule(a, info, inst_new, rng)
    mapped, forced = map_to_new_vms(a, info)
    if not forced.any(): return mapped
    if how == "random":
        mapped[forced] = rng.integers(0, inst_new.m, forced.sum()); return mapped
    keep = ~forced; ar = np.arange(inst_new.n)
    loads = np.bincount(mapped[keep], weights=inst_new.et[ar[keep], mapped[keep]], minlength=inst_new.m)
    for t in ar[forced][np.argsort(-inst_new.task_len[forced], kind="stable")]:
        v = int(np.argmin(loads + inst_new.et[t])); mapped[t] = v; loads[v] += inst_new.et[t, v]
    return mapped

def count_migrations(prev, new, info):
    """Voluntary migrations: tasks that existed in the previous epoch, still exist, whose VM survived, and that the
    new schedule moves. Returns (voluntary, forced, persistent_tasks)."""
    mapped, forced = map_to_new_vms(prev, info)
    persist = np.ones(len(new), bool)
    if info is not None and info["type"] == "churn": persist[info["idx"]] = False
    vol = persist & ~forced & (np.asarray(new) != mapped)
    return int(vol.sum()), int((persist & forced).sum()), int(persist.sum())

def incremental_list_schedule(prev, info, inst_new, rng):
    """Zero-voluntary-migration heuristic: persistent tasks stay; new (churned) and orphaned (failed-VM) tasks are placed
    longest first on the VM that finishes them earliest (Max-Min style on top of the existing loads)."""
    mapped, forced = map_to_new_vms(prev, info)
    place = forced.copy()
    if info is not None and info["type"] == "churn": place[info["idx"]] = True
    keep = ~place; ar = np.arange(inst_new.n)
    loads = np.bincount(mapped[keep], weights=inst_new.et[ar[keep], mapped[keep]], minlength=inst_new.m)
    for t in ar[place][np.argsort(-inst_new.task_len[place], kind="stable")]:
        v = int(np.argmin(loads + inst_new.et[t])); mapped[t] = v; loads[v] += inst_new.et[t, v]
    return mapped

# --- strategies --------------------------------------------------------------------------------------------
def run_dynamic(seq, algo, strategy, budget0, budget, seed, P=30, decoherence_c=0.5, mode="born_signed", gamma_shock=0.5,
                algo_kw=None, carry_elite=False, repair="random", mig_lambda=None, decoherence_by_event=None):
    """algo in {'QI-MRFO','QI-DMO','MRFO','DMO','GA','PSO'} or the per-epoch heuristics {'Max-Min' (full recompute),
    'Incremental' (zero voluntary migration)}; strategy in {'restart','continue','continue_struct','shock','hypermut'}.
    V5 options: algo_kw = extra optimizer kwargs (e.g. {'exchange': 1.0}); carry_elite = the previous epoch's best
    schedule, repaired for the change, seeds the register swarm (QI-MRFO); repair = 'random' | 'greedy' treatment of the
    tasks of a failed VM, or 'incremental' (H12: failed-VM and new churn tasks placed by list scheduling), used for the
    carried elite and the carried population of the GA / (1+1)-EA.
    mig_lambda (V5, H8): after the first epoch every optimizer sees the migration-aware objective
    makespan x (1 + mig_lambda * voluntary migrations / eligible tasks) relative to the previous deployed schedule;
    'Chooser' deploys whichever of Max-Min (recompute) and Incremental is cheaper under that objective.
    decoherence_by_event (V5, H13): {event type: c} overriding decoherence_c after that type of change (epochs e > 0,
    QI-MRFO and the (1+1)-EA); types not in the dict, and epoch 0, keep decoherence_c.
    Returns per-epoch dict: best makespan, LB, MaxMin makespan, normalised area under the best-so-far gap curve, and the
    voluntary / forced migrations of the deployed (best) schedule relative to the previous epoch's."""
    rng = np.random.default_rng(seed + 7)
    kw = dict(algo_kw or {})
    state = None; out = []; prev_best = None
    for e, (inst, info) in enumerate(seq):
        obj = Objective(inst); B = budget0 if e == 0 else budget
        n, m = inst.n, inst.m
        c_e = decoherence_c if (e == 0 or not decoherence_by_event) else decoherence_by_event.get(info["type"], decoherence_c)
        if mig_lambda is not None and e > 0:
            ref, forced_ = map_to_new_vms(prev_best, info)
            elig = ~forced_
            if info["type"] == "churn": elig[info["idx"]] = False
            obj = Objective(inst, kind="makespan_migration", ref=ref, mig_mask=elig, lam=mig_lambda)
        init = None
        if e > 0 and strategy != "restart":
            if algo in ("QI-MRFO", "QI-DMO"):
                init = adapt_register_state(state, info, mode, rng, struct=(strategy != "continue"))
                if strategy == "shock": init = shock_state(init, gamma_shock, mode)
                # carry_elite: True = always; "except_vm_add" (V5, H11) = event-aware, no elite after a VM addition, where
                # the carried schedule leaves the new VM empty and anchors the swarm away from it (H6b, H10a)
                use_elite = carry_elite is True or (carry_elite == "except_vm_add" and info["type"] != "vm_add")
                if carry_elite and algo != "QI-MRFO": raise ValueError("carry_elite is implemented for QI-MRFO only")
                if use_elite:
                    init = dict(init); init["elite"] = repair_schedule(prev_best, info, inst, rng, repair)
            elif algo in ("MRFO", "DMO", "PSO"):
                init = adapt_position_state(state, info, m, rng)
            elif algo in ("GA", "(1+1)-EA"):                # the (1+1)-EA carries its single schedule like a GA of size 1
                init = adapt_ga_state(state, info, m, rng, inst=inst, repair=repair)
        if algo in ("Max-Min", "Incremental", "Chooser"):
            if algo == "Max-Min" or e == 0: a = max_min(inst); f = obj(a)
            elif algo == "Incremental": a = incremental_list_schedule(prev_best, info, inst, rng); f = obj(a)
            else:                                   # Chooser: the cheaper of the two heuristics under the actual objective
                cands = [max_min(inst), incremental_list_schedule(prev_best, info, inst, rng)]
                fs = [obj(c) for c in cands]; k = int(np.argmin(fs)); a, f = cands[k], fs[k]
            tr = Tracker(n, m); tr.snapshot(obj.n_evals, [a], [f], f)
            r = {"best_f": f, "best_assign": a, "tracker": tr, "state": None}
        elif algo == "QI-MRFO": r = run_qimrfo(inst, obj, B, P=P, seed=seed * 100 + e, decoherence=c_e / n, mode=mode, init_state=init, track=False, **kw)
        elif algo == "QI-DMO": r = run_qidmo(inst, obj, B, P=P, seed=seed * 100 + e, decoherence=0.25 / n, mode=mode, init_state=init, track=False, **kw)
        elif algo == "MRFO": r = run_mrfo(inst, obj, B, P=P, seed=seed * 100 + e, init_state=init, track=False, **kw)
        elif algo == "DMO": r = run_dmo(inst, obj, B, P=P, seed=seed * 100 + e, init_state=init, track=False, **kw)
        elif algo == "PSO": r = run_pso(inst, obj, B, P=P, seed=seed * 100 + e, init_state=init, track=False, **kw)
        elif algo == "GA": r = run_ga(inst, obj, B, P=P, seed=seed * 100 + e, init_state=init, track=False, hypermutation=(0.2 if strategy == "hypermut" else 0.0), **kw)
        elif algo == "(1+1)-EA": r = run_one_plus_one(inst, obj, B, seed=seed * 100 + e, decoherence=c_e / n, init_state=init, track=False, **kw)
        else: raise ValueError(algo)
        state = r["state"]; tr = r["tracker"]
        lb = inst.lower_bound(); mm = obj._raw(max_min(inst))[0]
        ev = np.array(tr.evals, float); bs = np.array(tr.best, float)
        auc = float((getattr(np, 'trapezoid', None) or np.trapz)((bs - lb) / lb, ev) / max(1e-9, ev[-1] - ev[0])) if len(ev) > 1 else float((bs[-1] - lb) / lb)
        vol, forced, persist = count_migrations(prev_best, r["best_assign"], info) if e > 0 else (0, 0, n)
        ms_dep = float(r["best_f"]) if obj.kind == "makespan" else float(Objective(inst)._raw(r["best_assign"])[0])
        out.append({"epoch": e, "best": ms_dep, "lb": lb, "maxmin": mm, "gap": (ms_dep - lb) / lb, "auc_gap": auc, "m": m,
                    "type": None if info is None else info["type"], "migrations": vol, "forced": forced, "persist": persist,
                    "evals": obj.n_evals, "assign": np.asarray(r["best_assign"]).copy(),
                    "cost": float(r["best_f"]), "cost_gap": (float(r["best_f"]) - lb) / lb})
        prev_best = np.asarray(r["best_assign"]).copy()
    return out

DYN_CONFIGS = [("QI-MRFO", "restart", {}), ("QI-MRFO", "continue", {}), ("QI-MRFO", "shock", {"gamma_shock": 0.25}), ("QI-MRFO", "shock", {"gamma_shock": 0.5}), ("QI-MRFO", "shock", {"gamma_shock": 0.75}),
               ("MRFO", "restart", {}), ("MRFO", "continue", {}), ("GA", "restart", {}), ("GA", "continue", {}), ("GA", "hypermut", {})]
DYN_CHANGES = ["churn", "vm_fail", "vm_add", "drift"]

def run_dyn_job(job):
    change, algo, strat, kw, s = job
    seq = make_dynamic_sequence(50, 10, seed=s, K=5, change=change)
    out = run_dynamic(seq, algo, strat, CFG["dyn_budget0"], CFG["dyn_budget"], seed=s, P=CFG["P"], decoherence_c=CFG["gamma_c_mrfo"], **kw)
    post = out[1:]
    return {"change": change, "algo": algo, "strategy": strat + (f" g={kw['gamma_shock']}" if kw else ""), "seed": s,
            "post_gap": np.mean([o["gap"] for o in post]), "auc_gap": np.mean([o["auc_gap"] for o in post]), "vs_maxmin": np.mean([o["best"] / o["maxmin"] for o in post]),
            "epoch0_gap": out[0]["gap"]}

dyn_jobs = [(ch, a, st, kw, s) for ch in DYN_CHANGES for a, st, kw in DYN_CONFIGS for s in CFG["dyn_seeds"]]
t0 = time.time(); dyn_rows = None
if CFG["parallel"]:
    try:
        import multiprocessing as mp
        if mp.get_start_method(allow_none=True) in (None, "fork") and hasattr(os, "fork"):
            from concurrent.futures import ProcessPoolExecutor
            with ProcessPoolExecutor(max_workers=min(os.cpu_count() or 1, len(dyn_jobs)), mp_context=mp.get_context("fork")) as ex:
                dyn_rows = list(ex.map(run_dyn_job, dyn_jobs, chunksize=1))
    except Exception as e:
        print("parallel execution unavailable ->", type(e).__name__, "; running sequentially"); dyn_rows = None
if dyn_rows is None:
    dyn_rows = [run_dyn_job(j) for j in dyn_jobs]
print(f"dynamic pilot: {len(dyn_jobs)} runs in {time.time() - t0:.0f}s")
dyn = pd.DataFrame(dyn_rows); dyn.to_csv(os.path.join(CFG["results_dir"], f"dynamic_{MODE}.csv"), index=False)
dyn_tab = dyn.groupby(["change", "algo", "strategy"]).agg(post_gap_pct=("post_gap", lambda g: 100 * g.mean()), post_gap_sd=("post_gap", lambda g: 100 * g.std()), auc_gap_pct=("auc_gap", lambda g: 100 * g.mean()), vs_maxmin=("vs_maxmin", "mean")).round(2)
dyn_tab

In [ ]:
fig, axes = plt.subplots(1, len(DYN_CHANGES), figsize=(4.0 * len(DYN_CHANGES), 3.4), squeeze=False)
for ax, change in zip(axes[0], DYN_CHANGES):
    sub = dyn[dyn.change == change].groupby(["algo", "strategy"]).auc_gap.mean().reset_index()
    labels = [f"{a}\n{s}" for a, s in zip(sub.algo, sub.strategy)]
    ax.barh(range(len(sub)), sub.auc_gap * 100, color=[color_of(a) for a in sub.algo], linewidth=0, height=0.7)
    ax.set_yticks(range(len(sub))); ax.set_yticklabels(labels, fontsize=6.5); ax.invert_yaxis()
    style_axes(ax, f"{change}: area under the gap curve after changes (%)", "AUC of gap (%) — lower is faster recovery", "")
fig.tight_layout(); plt.savefig(os.path.join(CFG["results_dir"], f"fig_dynamic_{MODE}.png"), dpi=130); plt.show()

## SECTION 18 — Interpretation

The cell below prints the key comparisons from the results produced *in this execution*; the text that follows interprets the
pattern that the pilot study (5 seeds, 20 000 evaluations, documented in `lab_log.md` and the report) established and that the
`fast`/`full` modes re-test. Read the printed numbers first: if they contradict the text, the numbers win.

In [ ]:
def _g(df, algo, inst_name):
    s = df[(df.algo == algo) & (df.instance == inst_name)].gap; return 100 * s.mean() if len(s) else np.nan
print("=== Baseline: mean gap to LB (%) ===")
for inst_name in insts:
    print(f"{inst_name:26s} " + "  ".join(f"{a}={_g(baseline, a, inst_name):6.2f}" for a in ["MRFO", "QI-MRFO", "P-MRFO (linear twin)", "GA", "Max-Min", "DMO", "QI-DMO", "Random"]))
print("\n=== Mechanism diagnostics on the largest instance ===")
for a in ["MRFO", "QI-MRFO", "DMO", "QI-DMO", "GA"]:
    sub = baseline[(baseline.algo == a) & (baseline.instance == big)]
    print(f"{a:10s} move size early/late = {sub.move_sizes_early.mean():5.1f}/{sub.move_sizes_late.mean():5.1f}   wasted = {sub.wasted.mean():.3f}   end diversity = {sub.div_end.mean():.3f}   gb-improvements/1k evals = {sub.gb_impr_per_1k.mean():.2f}")
print("\n=== Ablation (mean gap %, averaged over ablation instances) ===")
print(abl_pivot.mean(1).round(2).to_string())
print("\n=== Dynamic pilot: mean post-change gap (%) / AUC (%) ===")
for (ch, a, s), g in dyn.groupby(["change", "algo", "strategy"]):
    print(f"{ch:8s} {a:8s} {s:16s} gap={100 * g.post_gap.mean():6.2f}  auc={100 * g.auc_gap.mean():6.2f}")

### Reading the results (pattern established in the pilot; verify against the printout above)

1. **The classical failure mode is the encoding, not the metaphor.** With the floor encoding, MRFO and DMO change ~half of all task assignments per candidate and fall to random-search level for $n \ge 50$; a discrete GA with 1/n mutation does not. This is the *observation* that motivated the mechanism.
2. **Superposition + measurement fixes it.** QI-MRFO/QI-DMO make move size a function of register purity; the gap to the lower bound at $n=100$ falls from ~100 % (MRFO) to ~1 % (QI-MRFO). The gain grows with $n$ as predicted.
3. **Decoherence is necessary and must be weak.** Without the channel the registers collapse (purity → 1) and 25–75 % of evaluations re-measure the same schedule. A depolarising strength $\gamma \approx (0.25\text{–}1)/n$ removes the waste and improves every instance; larger $\gamma$ degrades monotonically (dose–response). The optimum is host-dependent (MRFO tolerates more than DMO because it is greedy in every phase).
4. **The quantum-specific part is not what carries the effect.** The linear-probability twin matches the Born-rule version at the end of the run; signed vs unsigned amplitudes are indistinguishable. The Born rule only accelerates concentration (higher purity early). The honest statement is therefore: *a quantum-inspired representation (superposition, measurement, decoherence) repairs a real defect of swarm schedulers; its classical equivalent (a probability-vector swarm with a mutation floor) works equally well.*
5. **Collapse-conditioned attraction is essential.** Using the alpha's superposition instead of its measured schedule as attractor destroys the search (purity stays at 1/m).
6. **Boundary of usefulness.** On a static makespan batch the Max-Min list heuristic remains competitive or better on tall-barrier (bimodal) instances; with $m=2$ or trivial landscapes every method converges; the practical case for the population method is where heuristics do not apply: additive/multi-objective objectives and dynamic re-optimisation.
7. **Dynamic workloads (pilot).** Carrying the register state across a change beats restarting on every change type (recovery AUC 2–4× lower) and beats the GA's carried population on VM drift, VM addition and VM failure, because the representation has clean structural rules (column deletion = projective measurement, uniform share for a new VM, uniform registers for new tasks). The decoherence *shock* — the strong form of the "controlled forgetting" hypothesis — is **not** supported at 20 % churn / mild drift: $\gamma_{shock}=0.5$ gives at most a small final-gap gain at the cost of slower recovery, and shocks hurt on structural changes; GA hypermutation (its classical analogue) hurts everywhere. The open question is whether forgetting pays off only above a change-severity threshold (the report's next experiment).
8. **Overhead.** The register representation costs roughly $m$ times the memory and ~2× wall time per evaluation of the floor encoding at $n=100, m=10$; negligible against evaluation costs in a real simulator.

**Decision (see the report):** PROCEED with a *revised* framing — the research question is not "does the quantum metaphor beat the classical one?" (it does not) but "which properties of a measurement-based (superposition) schedule representation — purity-controlled move size, structural adaptation rules, and severity-dependent forgetting — matter for cloud re-scheduling under change, and where is the boundary?"

## SECTION 19 — V5: where the evaluations go, and critical exchange measurement (hypothesis H5)

**Observation (V5, `observe_v5_diagnostics.py`, `observe_v5_localopt.py`; reproduced below on this run's baseline).**
Four facts about QI-MRFO with $c=1$:
* **O1.** 31–46 % of its evaluations re-evaluate a schedule already seen in the run. The earlier "wasted" metric only
  counted parent-identical candidates.
* **O2.** A candidate can lower the makespan only if it moves a task off its reference schedule's critical VM. Every
  improving candidate does this, but only 20–45 % of candidates do.
* **O3.** The global best stops improving early, at 16–54 % of the budget.
* **O4.** QI-MRFO's end points are always *relocation-optimal*, yet 1–50 strictly improving critical **swaps** remain.

The product-state measurement samples tasks independently and so almost never produces the correlated two-task
change a relocation-optimal schedule needs.

**Mechanism (H5, `qi_core.critical_exchange`, `run_qimrfo(exchange=p_x)`).** With probability $p_x$ a measured candidate
$a$ additionally undergoes a *critical exchange*. A task $t$ is drawn uniformly from the critical VM $b$ of $a$, and a
task $u$ uniformly from the tasks on other VMs that are shorter than $t$ (necessary for $\mathrm{Load}_b$ to fall).
Then $a_t \leftrightarrow a_u$; if no shorter task exists, $t$ is relocated. The two registers then collapse onto the
outcome, $\Psi[t] \leftarrow \mathcal{D}_\gamma(E(a_t))$ and $\Psi[u] \leftarrow \mathcal{D}_\gamma(E(a_u))$, so an
accepted register remembers the exchange. Quantum reading: a *correlated* (non-product) measurement of a register pair,
followed by measurement back-action. Classical equivalent: swap mutation. The GA gets the identical operator
(`run_ga(exchange=p_x)`) as the control that decides whether the gain is specific to the register swarm.

**Protocol (pre-registered in `research_plan_v5.md`).** $p_x$ was tuned on the pilot instances only (development set,
selection-biased), then frozen at $p_x = 1$ and tested on 8 new families × 10 new instances (seeds 101–110) × 2 run
seeds at 20 000 evaluations. The committed results are `results/h5_tune/`, `results/h5_test/` and
`results/h5_analysis.md`. The cells below (i) reproduce the O1–O4 diagnostics on this run's baseline, (ii) run a small
held-out demonstration sized by `QI_MODE`, and (iii) print the committed held-out results when present.

In [ ]:
# (i) O1-O4 on this execution's baseline runs (Section 12)
diag_cols = ["gap", "gap2", "dup_global", "wasted", "touch_crit", "last_gb_impr_frac", "late_improving", "end_impr_reloc", "end_impr_swap"]
print("V5 diagnostics on the Section-12 baseline (means over seeds):")
baseline[baseline.algo.isin(["QI-MRFO", "P-MRFO (linear twin)", "GA", "Max-Min"])].groupby(["instance", "algo"])[diag_cols].mean().round(4)

In [ ]:
# (ii) held-out demonstration: new instances (inst_seed 101), CXM on QI-MRFO, its linear twin and the GA control
REGISTRY["QI-MRFO+CXM"] = _mk(run_qimrfo, P=CFG["P"], exchange=1.0, decoherence=lambda inst: CFG["gamma_c_mrfo"] / inst.n)
REGISTRY["P-MRFO+CXM (linear twin)"] = _mk(run_qimrfo, P=CFG["P"], mode="linear", exchange=1.0, decoherence=lambda inst: CFG["gamma_c_mrfo"] / inst.n)
REGISTRY["GA+CXM"] = _mk(run_ga, P=CFG["P"], exchange=1.0)
# H7 control: a (1+1)-EA with exactly the moves of a collapsed QI-MRFO+CXM (c and p_x tuned on the pilot instances only)
REGISTRY["(1+1)-EA+CXM"] = _mk(run_one_plus_one, decoherence=lambda inst: 1.0 / inst.n, exchange=0.5)
PALETTE.update({"QI-MRFO+CXM": "#0b4f9c", "GA+CXM": "#0e7a52", "P-MRFO+CXM (linear twin)": "#2e2370"})
H5_SPECS = {"smoke": [(80, 8, "uniform", "high"), (100, 10, "bimodal", "high")],
            "fast": [(80, 8, "uniform", "high"), (100, 10, "bimodal", "high"), (120, 12, "lognormal", "high"), (60, 12, "uniform", "low")],
            "full": [(80, 8, "uniform", "high"), (150, 15, "uniform", "high"), (100, 10, "bimodal", "high"), (200, 10, "bimodal", "none"),
                     (120, 12, "lognormal", "high"), (60, 12, "uniform", "low"), (100, 20, "lognormal", "low")]}[MODE]
H5_ALGOS = ["Max-Min", "GA", "GA+CXM", "(1+1)-EA+CXM", "QI-MRFO", "QI-MRFO+CXM", "P-MRFO+CXM (linear twin)"]
h5 = run_suite(H5_ALGOS, H5_SPECS, CFG["seeds"], CFG["budget"], inst_seed=101, label="H5 demo (inst_seed 101)")
h5.drop(columns=[c for c in h5.columns if c.startswith("curve_")]).to_csv(os.path.join(CFG["results_dir"], f"h5_demo_{MODE}.csv"), index=False)
print("gap to the preemptive LB (%), mean over seeds:")
display((h5.pivot_table(index="instance", columns="algo", values="gap2", aggfunc="mean")[H5_ALGOS] * 100).round(3))
h5.groupby("algo")[["end_impr_swap", "last_gb_impr_frac", "x_success", "dup_global", "move_size", "runtime_s"]].mean().reindex(H5_ALGOS).round(4)

In [ ]:
# (iii) the committed held-out experiment (80 new instances x 2 seeds), if the repository's results/ folder is present
_h5p = os.path.join("results", "h5_test", "records.csv")
if os.path.exists(_h5p):
    h5t = pd.read_csv(_h5p)
    _alg = ["Max-Min", "GA", "GA+CXM", "P-MRFO", "P-MRFO+CXM", "QI-MRFO", "QI-MRFO+CXM"]
    print("committed H5 held-out results: mean gap to the preemptive LB (%)")
    display((h5t.pivot_table(index="family", columns="algo", values="gap2", aggfunc="mean")[_alg] * 100).round(3))
    _w = h5t[h5t.algo.isin(["QI-MRFO", "QI-MRFO+CXM"])].groupby(["family", "inst_seed", "algo"]).gap2.mean().unstack("algo")
    _d = 100 * (_w["QI-MRFO+CXM"] - _w["QI-MRFO"]).values
    print(f"P1 (pooled over {len(_d)} instances): mean diff = {_d.mean():.3f} pp, 95% bootstrap CI = {np.round(boot_ci(_d), 3)}, "
          f"Wilcoxon p = {stats.wilcoxon(_d).pvalue:.2e}, CXM better on {(_d < 0).sum()}/{len(_d)} instances")
else:
    print("results/h5_test/records.csv not found (run `python exp_h5_cxm.py tune test analyze` in the repository)")

### 19.1 What the held-out experiment showed (committed run; `results/h5_analysis.md`, `results/h5_posthoc.md`)

The printout above is authoritative; where it disagrees with this text, the numbers win.

* **P1 (primary) confirmed.** CXM lowers QI-MRFO's gap to the preemptive bound from 3.40 % to 0.54 %: −2.86 pp,
  95 % CI [−3.58, −2.21], better on 76/80 new instances. Seven of 8 families are Holm-significant, and no family is
  worse. With $p_x = 0.5$ the result is the same (−2.81 pp).
* **Against Max-Min.** Unchanged QI-MRFO loses on 73/80 instances. QI-MRFO+CXM wins on 6 of 8 families and loses
  on n100 m20 lognormal/low. There Max-Min attains the lower bound on every instance: the largest task alone on the
  fastest VM is provably optimal, and reaching it by local moves needs makespan-neutral steps that strict acceptance
  rejects (a plateau).
* **P5.** CXM is *not* a generic fix. The GA gains 0.81 pp, and QI-MRFO+CXM beats GA+CXM on 73/80 instances. The
  exchange works because the register swarm applies it greedily around a collapsed best and stores it through
  back-action.
* **P4 falsified.** Under CXM the classical linear twin is slightly but significantly **better** than the Born-rule
  version (68/80 instances, margins below 0.2 pp). The quantum-specific ingredient is again not a source of advantage.
* **P3 partially supported.** The end points are nearly swap-optimal (improving swaps 140 → 11) and the last
  improvement comes later (54 % → 76 % of the budget). The late-half improving rate fell because CXM converges early.
* **P2** (absolute dose–response) not supported; the post hoc relative version is ρ = 0.71 and is exploratory.

In [ ]:
# (iv) committed H7 run (fresh instances, seeds 201-210): is the register swarm needed once CXM exists? Max-Min seeding
_h7p = os.path.join("results", "h7_test", "records.csv")
if os.path.exists(_h7p):
    h7t = pd.read_csv(_h7p)
    _alg7 = ["Max-Min", "GA+CXM+seed", "(1+1)-EA+CXM", "(1+1)-EA+CXM+seed", "P-MRFO+CXM", "P-MRFO+CXM+seed", "QI-MRFO+CXM", "QI-MRFO+CXM+seed"]
    print("committed H7 results: mean gap to the preemptive LB (%)")
    display((h7t.pivot_table(index="family", columns="algo", values="gap2", aggfunc="mean")[_alg7] * 100).round(3))
    for _a, _b in [("QI-MRFO+CXM", "(1+1)-EA+CXM"), ("QI-MRFO+CXM+seed", "QI-MRFO+CXM")]:
        _w = h7t[h7t.algo.isin([_a, _b])].groupby(["family", "inst_seed", "algo"]).gap2.mean().unstack("algo")
        _d = 100 * (_w[_a] - _w[_b]).values
        print(f"{_a} - {_b}: mean {_d.mean():.3f} pp, CI {np.round(boot_ci(_d), 3)}, Wilcoxon p = {stats.wilcoxon(_d).pvalue:.2e}, {_a} better on {(_d < 0).sum()}/{len(_d)}")
else:
    print("results/h7_test/records.csv not found (run `python exp_h7_swarm_seed.py tune test analyze` in the repository)")

### 19.2 H7: the swarm vs a (1+1)-EA with the same moves, and Max-Min seeding (`results/h7_analysis.md`)

* **Pre-registered primary H7a failed.** The (1+1)-EA with identical moves beats QI-MRFO+CXM on 58/80 fresh instances,
  and no family favours the swarm. For static makespan the exchange measurement, not the register swarm, carries the
  gain.
* **Seeding (H7b) confirmed.** A Max-Min basis state fixes the big-task plateau (3.11 % → 0.02 %) and harms nothing.
* **Best static method tested.** Max-Min seed + (1+1)-EA with CXM moves (mean rank 2.70 of 8).
* **What remains for the swarm.** Its case is re-optimisation under change (Section 20).
* **The decoherence floor once CXM exists** (descriptive, development set; `results/v5_c_sweep_paired.md`).
  * Without CXM, c = 0 multiplies the gap by 2.6–4.3.
  * With CXM, the gap is flat for c ∈ [0, 2]; c then only sets duplicate evaluations.

## SECTION 20 — V5: re-optimisation under change with migration cost (hypothesis H6)

Section 17h compared strategies by the makespan after each change. A running cloud also pays for every task that the
new schedule **migrates** away from the previous deployed one. V5 therefore counts *voluntary migrations*: persistent
tasks whose VM survived but that the new schedule moves. Replaced tasks and tasks of a failed VM are excluded.
`qi_dynamic.run_dynamic` gains four things:
* optimizer kwargs, so CXM can be used under change;
* **elite carry-over**: the previous best schedule, repaired for the change, seeds the register swarm (fixing audit risk
  R4: the GA always carried its elite, QI-MRFO did not);
* greedy repair of the tasks of a failed VM;
* two heuristic competitors: **Max-Min recomputed every epoch** (unlimited migration) and an **incremental list
  heuristic** (zero voluntary migration).

The pre-registered design is `research_plan_v5.md` §11. The committed run is `results/h6_dynamic/` with its analysis
in `results/h6_analysis.md`. The cells below run a small demonstration sized by `QI_MODE` and print the committed
results when present.

In [ ]:
# small held-out dynamic demonstration (instance seed 101): post-change gap, recovery AUC and voluntary migrations
DYN5 = {"Max-Min (recompute)": dict(algo="Max-Min"), "Incremental (no migration)": dict(algo="Incremental"),
        "GA+CXM continue": dict(algo="GA", strategy="continue", repair="greedy", algo_kw={"exchange": 1.0}),
        "QI-MRFO continue": dict(algo="QI-MRFO", strategy="continue_struct", repair="greedy"),
        "QI-MRFO+CXM continue+elite": dict(algo="QI-MRFO", strategy="continue_struct", repair="greedy", carry_elite=True, algo_kw={"exchange": 1.0})}
_dyn_n, _dyn_changes = (60, ["mixed"]) if MODE == "smoke" else (100, ["churn", "drift", "mixed"])
rows5 = []
for ch in _dyn_changes:
    seq5 = make_dynamic_sequence(_dyn_n, 10, seed=101, K=4, change=ch)
    for name, spec5 in DYN5.items():
        kw = dict(spec5)                                  # copy: the specs are reused for every change type
        o = run_dynamic(seq5, kw.pop("algo"), kw.pop("strategy", "continue_struct"), CFG["dyn_budget0"], CFG["dyn_budget"], seed=101,
                        P=CFG["P"], decoherence_c=CFG["gamma_c_mrfo"], **kw)
        post = o[1:]
        rows5.append({"change": ch, "strategy": name, "post_gap_%": 100 * np.mean([x["gap"] for x in post]),
                      "auc_%": 100 * np.mean([x["auc_gap"] for x in post]), "migrations/epoch": np.mean([x["migrations"] for x in post])})
pd.DataFrame(rows5).round(3)

In [ ]:
_h6p = os.path.join("results", "h6_dynamic", "records.csv")
if os.path.exists(_h6p):
    h6 = pd.read_csv(_h6p)
    for metric, lab, sc in [("post_gap", "post-change gap (%)", 100), ("post_auc", "recovery AUC (%)", 100), ("migrations", "voluntary migrations per epoch", 1)]:
        print(f"committed H6 results — {lab}:")
        display((h6.pivot_table(index="scenario", columns="algo", values=metric, aggfunc="mean") * sc).round(3))
else:
    print("results/h6_dynamic/records.csv not found (run `python exp_h6_dynamic.py run analyze` in the repository)")

### 20.1 What the committed H6 run showed (`results/h6_analysis.md`; the printout above is authoritative)

* **CXM under change (H6a, primary) confirmed.** The post-change gap drops by 2.98 pp on 60/60 scenario-seed pairs.
* **Elite carry-over (H6b) rejected as a default.** It helps after churn and VM failure but hurts after VM addition:
  the carried schedule leaves the new VM empty and becomes the attractor, and an exchange cannot fill an empty VM.
* **Against recomputing Max-Min every epoch (H6d).** The carried-state swarm with CXM reaches a lower gap (−0.33 pp,
  48/60) with 41 % fewer voluntary migrations. The exception is heavy-tailed n = 200, where Max-Min is better.
* **Against the GA.** Carried-state QI-MRFO+CXM beats GA+CXM on 60/60, and CXM makes the GA worse under change.
* **Quantum-specific part.** The classical linear twin is again better than the Born rule (55/60).
* **Open cost.** CXM triples migrations, because the objective ignores them. A migration-aware objective is the next
  hypothesis.

In [ ]:
# committed migration-priced runs: H8 (seeds 201-210), H10 elite anchor (301-310), H11 event-aware elite (401-410),
# H12 incremental elite (501-510), H13 event-aware decoherence (601-610)
for _exp, _lab in [("h8_migration", "H8"), ("h10_elite_migration", "H10"), ("h11_event_elite", "H11"), ("h12_incremental_elite", "H12"),
                   ("h13_event_gamma", "H13")]:
    _p = os.path.join("results", _exp, "records.csv")
    if not os.path.exists(_p):
        print(f"{_p} not found (run the corresponding exp_*.py in the repository)"); continue
    _d = pd.read_csv(_p)
    print(f"{_lab}: pooled post-change cost gap (%), makespan gap (%) and voluntary migrations per epoch")
    display((_d.groupby(["mig_lambda", "algo"])[["post_cost_gap", "post_gap"]].mean() * 100).round(3).join(
        _d.groupby(["mig_lambda", "algo"])[["migrations"]].mean().round(1)))

### 20.2 H8: migrations priced (`results/h8_analysis.md`)

* **Setup.** After each change every method optimises makespan × (1 + λ · migrations / eligible tasks).
* **Against the best heuristic chooser.** The carried-state swarm with CXM wins pooled at λ = 0.2 and λ = 1.0 but not
  at λ = 0.05. The effect depends on the change type: the swarm wins where change calls for coordinated
  reconfiguration (drift, mixed events, VM addition) and loses where zero-migration repair is near-optimal (churn, VM
  failure).
* **Against the (1+1)-EA with the same moves.** It cannot recover from VM additions: every single-task move onto the
  new VM is uphill under the price. The register swarm's uniform-share rule for new VMs produces multi-task moves that
  escape.
* **What H10 tests next.** Anchoring the swarm on the deployed schedule (the elite) for local changes.

### 20.3 H9–H13 (`results/h9_analysis.md`, `h10_analysis.md`, `h11_analysis.md`, `h12_analysis.md`, `h13_analysis.md`)

* **H9: purity-regulated decoherence (the original report's §21 proposal) is falsified.** It raises duplicate
  evaluations (27 % → 46 %) and worsens the gap. Mean purity is dominated by a few diffuse registers, so the controller
  lowers γ exactly where the collapsed registers needed it.
* **H10: an unconditional elite is rejected.** It wins every churn and VM-failure pair but loses badly after VM
  additions, where the carried schedule leaves the new VM empty and anchors the swarm away from it.
* **H11: the event-aware elite (none after a VM addition) is confirmed on fresh seeds at every λ.** It beats the best
  heuristic chooser at every migration price. The remaining boundary is pure churn at λ ≥ 0.2, where zero-migration
  repair is best.
* **H12: the incremental elite.** After churn the elite used to leave each new task on the VM of the task it replaced.
  It now places new tasks by list scheduling.
  * It is better than H11 at λ = 0.2 and λ = 1.0. At λ = 0.05 it wins 26/3, but the CI of the mean includes 0, so it
    is not retained there.
  * It wins pure churn against the Chooser at λ ≤ 0.2.
  * Against incremental repair + a (1+1)-EA with the same moves, the swarm wins only through VM additions.
* **H13: no decoherence floor after local changes.** Under a migration price the floor's random re-draws are paid
  migrations.
  * It is better than c = 1 at λ = 0.2 and λ = 1.0.
  * Keeping c = 1 after VM additions made no difference.
  * The final configuration beats the Chooser at every λ (58/2, 57/3, 59/1).

In [ ]:
# small demonstration of the migration-priced configurations (instance seed 101, lambda = 0.2): the Chooser,
# the H11 swarm (event-aware elite), the H12 swarm (event-aware incremental elite) and H13 (no floor after local changes)
DYN12 = {"Chooser (cheaper of the two)": dict(algo="Chooser"),
         "QI-MRFO+CXM event-aware elite (H11)": dict(algo="QI-MRFO", strategy="continue_struct", repair="greedy",
                                                     carry_elite="except_vm_add", algo_kw={"exchange": 1.0}),
         "QI-MRFO+CXM event-aware incremental elite (H12)": dict(algo="QI-MRFO", strategy="continue_struct", repair="incremental",
                                                                 carry_elite="except_vm_add", algo_kw={"exchange": 1.0}),
         "... + no floor after local changes (H13)": dict(algo="QI-MRFO", strategy="continue_struct", repair="incremental",
                                                          carry_elite="except_vm_add", algo_kw={"exchange": 1.0},
                                                          decoherence_by_event={"churn": 0.0, "drift": 0.0, "vm_fail": 0.0})}
_n12, _ch12 = (60, ["churn", "mixed"]) if MODE == "smoke" else (100, ["churn", "drift", "vm_add", "mixed"])
rows12 = []
for ch in _ch12:
    seq12 = make_dynamic_sequence(_n12, 10, seed=101, K=4, change=ch)
    for name, spec12 in DYN12.items():
        kw = dict(spec12)                                 # copy: the specs are reused for every change type
        o = run_dynamic(seq12, kw.pop("algo"), kw.pop("strategy", "continue_struct"), CFG["dyn_budget0"], CFG["dyn_budget"],
                        seed=101, P=CFG["P"], decoherence_c=1.0, mig_lambda=0.2, **kw)
        post = o[1:]
        rows12.append({"change": ch, "strategy": name, "cost_gap_%": 100 * np.mean([x["cost_gap"] for x in post]),
                       "makespan_gap_%": 100 * np.mean([x["gap"] for x in post]),
                       "migrations/epoch": np.mean([x["migrations"] for x in post])})
pd.DataFrame(rows12).round(3)